# EMiF Project — Has the structure of risk in financial markets changed since COVID-19?

## Section 0 — Data loading and transformations

This section prepares a clean dataset used by all subsequent sections. Every
transformation is documented and justified.

**Objects produced:**
- `prices`       — daily prices, forward-filled, full sample
- `yields`       — daily level of the two rate series
- `returns`      — daily log-returns, full sample, 12 assets (rates excluded)
- `ret_pre`      — log-returns, 2010–2020, main pre-COVID reference period
- `ret_post`     — log-returns, 2020–2026, post-COVID regime
- `ret_pre_full` — log-returns, 1990–2020, robustness checks only

---

### What do we mean by "structure of risk"?

In this project, the *structure of risk* refers to three distinct but interconnected dimensions:

1. **Risk levels** — the magnitude and persistence of volatility within each asset class,
   measured by GARCH(1,1) parameters (Section 2). Formally tested via Likelihood-Ratio
   tests comparing constrained (equal parameters pre/post) vs unconstrained models, and
   via Levene / Fisher variance equality tests.

2. **Risk co-movement** — correlations and covariance structure across assets, measured
   by rolling correlations and the DCC model (Section 3).

3. **Risk regimes** — whether changes are permanent (structural break) or recurring
   (regime switch), tested via Bai-Perron PELT algorithm and Markov-switching (Section 4).

A change in the structure of risk means that at least one of these three dimensions has
shifted significantly and durably after March 2020. Our answer is given by asset class:
Equities, Commodities, Credit and FX.

---

### On the choice of breakpoint

We set the COVID breakpoint at **23 March 2020**: the S&P 500 market bottom and the
Federal Reserve announcement of unlimited QE (VIX peaked above 80). Statistical validity
is confirmed by a CUSUM test on S&P 500 squared returns (Section 2) and by the
Bai-Perron/PELT endogenous break detection (Section 4).

### Why 2010–2020 as the pre-COVID reference period?

Using everything before March 2020 would include the Dot-com crash, 9/11, and the GFC —
events that are themselves major structural breaks. We use **2010–2020**, a decade of low
rates and subdued volatility, as the meaningful pre-COVID benchmark. The full 1990–2020
sample is kept as `ret_pre_full` for robustness only.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize, differential_evolution
from scipy.special import gammaln
from scipy.stats import norm, levene, f as f_dist, chi2, jarque_bera
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox
from statsmodels.tsa.stattools import adfuller, kpss
import statsmodels.api as sm
import ruptures as rpt
import warnings
warnings.filterwarnings('ignore')

# ── Global parameters ────────────────────────────────────────
DATA_PATH      = 'Data.xlsx'
SHEET          = 'Feuil1'
ANNUALIZATION  = 252
COVID_BREAK    = '2020-03-23'
PRE_START      = '2010-01-01'
PRE_START_FULL = '1990-01-01'
RATE_SERIES    = ['US T 10-year Yield', 'German Gov 10-year yield']

CATEGORIES = {
    'Equities'   : ['S&P500', 'Eurostoxx 50', 'Hang Seng', 'MSCI EM', 'SMI'],
    'Commodities': ['Oil futures', 'Gold'],
    'FX'         : ['EURUSD', 'USDJPY', 'USDCHF'],
    'Credit'     : ['US IG Bonds', 'US HY Bonds'],
}
COLORS = {'pre': 'steelblue', 'post': 'firebrick', 'full': 'dimgray'}

FIG_DIR = Path('Figures')
TAB_DIR = Path('Tables')
FIG_DIR.mkdir(exist_ok=True)
TAB_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})
print('Libraries and parameters loaded.')

### 0.1  Loading and cleaning the data

Missing values arise from calendar mismatches between markets. We forward-fill: the last
observed price is carried forward until the next available observation.

Oil futures prices turned negative in April 2020. Non-positive prices are treated as
missing and forward-filled before computing log-returns.

Log-returns: $r_{i,t} = \log(P_{i,t}) - \log(P_{i,t-1})$. Preferred over simple returns
because they are approximately stationary, time-additive and symmetric.

Rate series are stored as levels and analysed as first differences (yield changes).
Log-differencing yields (German yield was negative 2015–2020) produces economically
meaningless values and is avoided.

In [ ]:
def load_data(path, sheet, rate_series, pre_start, pre_start_full, covid_break):
    raw = pd.read_excel(path, sheet_name=sheet, index_col=0, parse_dates=True)
    raw = raw.sort_index().ffill()
    missing_before = (raw.isnull().sum() / len(raw) * 100).round(1)
    yields_ = raw[rate_series].copy()
    non_rate = [c for c in raw.columns if c not in rate_series]
    prices_ = raw[non_rate].where(raw[non_rate] > 0).ffill().dropna(how='all')
    returns_ = np.log(prices_).diff().dropna()
    ret_pre_ = returns_.loc[pre_start : pd.Timestamp(covid_break) - pd.Timedelta(days=1)]
    ret_post_ = returns_.loc[covid_break:]
    ret_pre_full_ = returns_.loc[pre_start_full : pd.Timestamp(covid_break) - pd.Timedelta(days=1)]
    return raw, yields_, prices_, returns_, ret_pre_, ret_post_, ret_pre_full_, missing_before

raw, yields, prices, returns, ret_pre, ret_post, ret_pre_full, missing_before = load_data(
    DATA_PATH, SHEET, RATE_SERIES, PRE_START, PRE_START_FULL, COVID_BREAK
)
print(f'Full sample   : {returns.index.min().date()} to {returns.index.max().date()}')
print(f'Pre-COVID     : {ret_pre.index.min().date()} to {ret_pre.index.max().date()}')
print(f'Post-COVID    : {ret_post.index.min().date()} to {ret_post.index.max().date()}')
print(f'Pre-COVID full: {ret_pre_full.index.min().date()} to {ret_pre_full.index.max().date()}')
print(f'Assets: {list(returns.columns)}')
print(f'German Gov yield — min: {yields["German Gov 10-year yield"].min():.4f}')

In [ ]:
print('Missing values before treatment (%):\n')
print(missing_before.to_string())
missing_before.to_csv(TAB_DIR / 'tab0_missing_values.csv')
print('\nAssets by category:')
for cat, assets in CATEGORIES.items():
    available = [a for a in assets if a in returns.columns]
    print(f'  {cat:15}: {", ".join(available)}')

### 0.3  Cumulative returns and yield levels

Vertical dashed line marks 23 March 2020. Rate series displayed as yield levels.

In [ ]:
for cat, assets in CATEGORIES.items():
    available = [a for a in assets if a in returns.columns]
    cum = (1 + returns[available]).cumprod()
    fig, ax = plt.subplots(figsize=(13, 4))
    cum.plot(ax=ax, linewidth=1)
    ax.axvline(pd.Timestamp(COVID_BREAK), color='firebrick', linestyle='--',
               linewidth=1.2, label='COVID break')
    ax.set_title(f'Cumulative returns — {cat}')
    ax.set_ylabel('Growth of 1')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'fig0_cumret_{cat.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

fig, ax = plt.subplots(figsize=(13, 4))
yields.plot(ax=ax, linewidth=1)
ax.axvline(pd.Timestamp(COVID_BREAK), color='firebrick', linestyle='--', linewidth=1.2)
ax.set_title('10-year government yields — level (%)')
ax.set_ylabel('Yield (%)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig0_yields_level.png', dpi=150, bbox_inches='tight')
plt.show()

### 0.4  Descriptive statistics by asset class

Four moments compared across periods: annualised volatility, skewness, excess kurtosis,
and minimum daily return.

We supplement the moment comparison with a **Jarque-Bera normality test**. Under H₀ the
return distribution is Gaussian; rejection (p < 0.05) confirms heavy tails and/or
asymmetry, motivating Student-t innovations in Section 2.

In [ ]:
def desc_stats(ret):
    jb_p = ret.apply(lambda s: jarque_bera(s.dropna())[1]).round(4)
    return pd.DataFrame({
        'Ann. Vol (%)'    : (ret.std() * np.sqrt(ANNUALIZATION) * 100).round(2),
        'Skewness'        : ret.apply(stats.skew).round(3),
        'Kurt. (excess)'  : ret.apply(stats.kurtosis).round(3),
        'Min return (%)'  : (ret.min() * 100).round(2),
        'JB p-value'      : jb_p,
        'Normal (5%)'     : jb_p.apply(lambda p: 'No' if p < 0.05 else 'Yes'),
    })

combined = pd.concat(
    [desc_stats(ret_pre), desc_stats(ret_post)], axis=1,
    keys=['Pre-COVID (2010-2020)', 'Post-COVID (2020-2026)']
)
combined.to_csv(TAB_DIR / 'tab0_descriptive_stats.csv')
combined

In [ ]:
ordered_assets, ordered_labels = [], []
for cat, assets in CATEGORIES.items():
    for a in assets:
        if a in returns.columns:
            ordered_assets.append(a)
            ordered_labels.append(f'{a}\n({cat})')

vol_pre_ord  = ret_pre[ordered_assets].std()  * np.sqrt(ANNUALIZATION) * 100
vol_post_ord = ret_post[ordered_assets].std() * np.sqrt(ANNUALIZATION) * 100

x, width = np.arange(len(ordered_assets)), 0.38
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width/2, vol_pre_ord,  width, label='Pre-COVID (2010-2020)',  color=COLORS['pre'])
ax.bar(x + width/2, vol_post_ord, width, label='Post-COVID (2020-2026)', color=COLORS['post'])

boundaries, count = [], 0
for cat, assets in CATEGORIES.items():
    count += len([a for a in assets if a in returns.columns])
    boundaries.append(count - 0.5)
for b in boundaries[:-1]:
    ax.axvline(b, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)

ax.set_xticks(x)
ax.set_xticklabels(ordered_labels, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Annualised volatility (%)')
ax.set_title('Annualised volatility — pre vs post-COVID (by asset class)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig0_vol_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 0.5  Stationarity of log-returns — ADF and KPSS tests

Before fitting any model, we verify that log-returns are stationary.

- **ADF** (Augmented Dickey-Fuller): H₀ = unit root (non-stationary). Rejection confirms stationarity.
- **KPSS** (Kwiatkowski-Phillips-Schmidt-Shin): H₀ = stationary. Failure to reject confirms stationarity.

We use automatic lag selection (AIC) for ADF and the asymptotic 5% critical values.
A series is declared stationary if ADF rejects H₀ **and** KPSS fails to reject H₀.

Note: USDCHF pre-COVID is expected to show weaker evidence of stationarity due to the
SNB peg (near-zero variance regime), which biases ADF toward non-rejection.

In [ ]:
stationarity_results = []
for asset in returns.columns:
    cat = next((c for c, lst in CATEGORIES.items() if asset in lst), 'Other')
    for label, ret in [('Full sample', returns), ('Pre-COVID', ret_pre), ('Post-COVID', ret_post)]:
        s = ret[asset].dropna()
        # ADF
        adf_stat, adf_p, _, _, adf_cv, _ = adfuller(s, autolag='AIC')
        adf_reject = adf_p < 0.05
        # KPSS (trend='c' = constant)
        try:
            kpss_stat, kpss_p, _, kpss_cv = kpss(s, regression='c', nlags='auto')
            kpss_reject = kpss_p < 0.05   # reject H0 of stationarity → non-stationary
        except Exception:
            kpss_stat, kpss_p, kpss_reject = np.nan, np.nan, False
        stationary = adf_reject and not kpss_reject
        stationarity_results.append({
            'Asset class': cat, 'Asset': asset, 'Period': label,
            'ADF stat'  : round(adf_stat, 3),
            'ADF p-val' : round(adf_p, 4),
            'ADF reject H0': adf_reject,
            'KPSS stat' : round(kpss_stat, 3) if not np.isnan(kpss_stat) else np.nan,
            'KPSS p-val': round(kpss_p, 4)    if not np.isnan(kpss_p)    else np.nan,
            'KPSS reject H0': kpss_reject,
            'Stationary (5%)': 'Yes ✓' if stationary else 'No ✗',
        })

stat_df = pd.DataFrame(stationarity_results).set_index(['Asset class', 'Asset', 'Period'])
stat_df.to_csv(TAB_DIR / 'tab0_stationarity.csv')
stat_df[['ADF stat', 'ADF p-val', 'ADF reject H0',
         'KPSS stat', 'KPSS p-val', 'KPSS reject H0', 'Stationary (5%)']]

---
## Section 1 — Stylized facts

Before estimating any model, we document the classical properties of financial returns
and test whether these properties have changed between the two periods. This section
provides a model-free first answer and motivates the GARCH estimation in Section 2.

The workflow covers: volatility proxies, autocorrelation structure, formal ARCH tests,
and excess kurtosis, all presented by asset class.

**Theoretical motivation.** Mandelbrot (1963) first documented fat tails and volatility
clustering in commodity prices; Fama (1965) confirmed these features for equity returns.
Engle (1982) formalised volatility clustering via the ARCH model, and Bollerslev (1986)
extended it to GARCH. The stylised facts below are consistent with this literature and
motivate our modelling choices in Section 2.

### 1.1  Simple volatility proxies

Three proxies for time-varying volatility on one representative asset per class:
- **Rolling 20-day volatility**: captures short-term fluctuations
- **Rolling 60-day volatility**: smoother medium-term trend
- **EWMA volatility (λ = 0.94)**: RiskMetrics standard (J.P. Morgan, 1996)

If volatility were constant, all three would be flat. Analysis is restricted to 2010–2026.

In [ ]:
def ewma_vol(r, lam=0.94):
    r = np.asarray(r)
    sigma2 = np.empty_like(r)
    sigma2[0] = np.var(r)
    for t in range(1, len(r)):
        sigma2[t] = lam * sigma2[t-1] + (1 - lam) * r[t-1]**2
    return np.sqrt(sigma2) * np.sqrt(ANNUALIZATION) * 100

representatives = {
    'Equities': 'S&P500', 'Commodities': 'Oil futures',
    'Credit': 'US HY Bonds', 'FX': 'EURUSD',
}
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
for idx, (cat, asset) in enumerate(representatives.items()):
    series = returns.loc[PRE_START:][asset].dropna()
    roll20 = series.rolling(20).std()  * np.sqrt(ANNUALIZATION) * 100
    roll60 = series.rolling(60).std()  * np.sqrt(ANNUALIZATION) * 100
    ewma   = pd.Series(ewma_vol(series.values), index=series.index)
    ax = axes[idx]
    roll20.plot(ax=ax, linewidth=0.8, label='Rolling 20d',  color='steelblue', alpha=0.8)
    roll60.plot(ax=ax, linewidth=1.0, label='Rolling 60d',  color='dimgray')
    ewma.plot(  ax=ax, linewidth=0.8, label='EWMA (λ=0.94)', color='firebrick', alpha=0.8)
    ax.axvline(pd.Timestamp(COVID_BREAK), color='black', linestyle='--',
               linewidth=1.2, label='COVID break')
    ax.set_title(f'{cat} — {asset}')
    ax.set_ylabel('Ann. volatility (%)')
    ax.legend(fontsize=7)
fig.suptitle('Simple volatility proxies by asset class', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_vol_proxies.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.2  Autocorrelation structure

Raw returns should have little autocorrelation if markets are approximately efficient
(Fama, 1970). Squared returns display significant positive autocorrelation if volatility
clustering is present — the core motivation for ARCH-type models (Engle, 1982).

In [ ]:
acf_summary = []
for asset in returns.columns:
    acf_pre  = pd.Series(ret_pre[asset].dropna()**2).autocorr(lag=1)
    acf_post = pd.Series(ret_post[asset].dropna()**2).autocorr(lag=1)
    cat = next((c for c, assets in CATEGORIES.items() if asset in assets), 'Other')
    acf_summary.append({
        'Asset class': cat, 'Asset': asset,
        'ACF(1) Pre-COVID' : round(acf_pre,  3),
        'ACF(1) Post-COVID': round(acf_post, 3),
        'Change': '↓' if acf_post < acf_pre else '↑',
    })
acf_df = pd.DataFrame(acf_summary).set_index(['Asset class', 'Asset'])
acf_df.to_csv(TAB_DIR / 'tab1_acf_summary.csv')
acf_df

### 1.3  Formal tests for ARCH effects

- **Ljung-Box test on squared returns**: tests autocorrelation of up to lag 10
- **ARCH-LM test** (Engle, 1982): LM test of no conditional heteroskedasticity

Rejecting both nulls confirms volatility clustering and justifies GARCH modelling.

In [ ]:
results = []
for asset in returns.columns:
    cat = next((c for c, lst in CATEGORIES.items() if asset in lst), 'Other')
    for label, ret in [('Pre-COVID', ret_pre), ('Post-COVID', ret_post)]:
        series = ret[asset].dropna()
        lb  = acorr_ljungbox(series**2, lags=[10], return_df=True)
        lm  = het_arch(series, nlags=10)
        results.append({
            'Asset class': cat, 'Asset': asset, 'Period': label,
            'LB stat'        : round(lb['lb_stat'].values[0], 2),
            'LB p-value'     : round(lb['lb_pvalue'].values[0], 4),
            'ARCH-LM stat'   : round(lm[0], 2),
            'ARCH-LM p-value': round(lm[1], 4),
        })
tests_df = pd.DataFrame(results).set_index(['Asset class', 'Asset', 'Period'])
tests_df.to_csv(TAB_DIR / 'tab1_arch_tests.csv')
tests_df

### 1.4  Excess kurtosis by asset class

Excess kurtosis measures tail fatness relative to a Gaussian. Returns are winsorised
at ±4σ to neutralise two isolated events: the SNB flash crash (Jan 2015, USDCHF) and
negative oil futures prices (Apr 2020). This winsorisation applies **only here** and
does not affect any other section.

Winsorising at ±4σ removes at most 0.006% of observations under normality, so the
procedure is conservative — it corrects for data anomalies without distorting the
structural properties of the distribution.

In [ ]:
def winsorised_kurtosis(series, threshold=4):
    mu, sigma = series.mean(), series.std()
    clipped = series.clip(lower=mu - threshold*sigma, upper=mu + threshold*sigma)
    return stats.kurtosis(clipped)

kurt_pre  = ret_pre.apply(winsorised_kurtosis)
kurt_post = ret_post.apply(winsorised_kurtosis)
kurt_df   = pd.DataFrame({
    'Pre-COVID (2010-2020)' : kurt_pre,
    'Post-COVID (2020-2026)': kurt_post,
})
ordered_assets, ordered_labels = [], []
for cat, assets in CATEGORIES.items():
    for a in assets:
        if a in kurt_df.index:
            ordered_assets.append(a)
            ordered_labels.append(f'{a}\n({cat})')
kurt_df = kurt_df.loc[ordered_assets]

fig, ax = plt.subplots(figsize=(14, 5))
x, width = np.arange(len(kurt_df)), 0.38
ax.bar(x - width/2, kurt_df['Pre-COVID (2010-2020)'],  width,
       label='Pre-COVID (2010-2020)',  color=COLORS['pre'])
ax.bar(x + width/2, kurt_df['Post-COVID (2020-2026)'], width,
       label='Post-COVID (2020-2026)', color=COLORS['post'])
boundaries, count = [], 0
for cat, assets in CATEGORIES.items():
    count += len([a for a in assets if a in kurt_df.index])
    boundaries.append(count - 0.5)
for b in boundaries[:-1]:
    ax.axvline(b, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(ordered_labels, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Excess kurtosis (winsorised at ±4σ)')
ax.set_title('Excess kurtosis — pre vs post-COVID (by asset class)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_kurtosis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
summary = []
for asset in returns.columns:
    for label, ret in [('Pre-COVID', ret_pre), ('Post-COVID', ret_post)]:
        series = ret[asset].dropna()
        lm = het_arch(series, nlags=10)
        jb_stat, jb_p = jarque_bera(series)
        summary.append({
            'Asset': asset, 'Period': label,
            'ACF(1) r²'      : round(pd.Series(series**2).autocorr(lag=1), 3),
            'Kurtosis'       : round(winsorised_kurtosis(series), 3),
            'ARCH-LM p-value': round(lm[1], 4),
            'JB p-value'     : round(jb_p, 4),
            'Normal (5%)'    : 'No' if jb_p < 0.05 else 'Yes',
        })
summary_df = pd.DataFrame(summary).set_index(['Asset', 'Period'])
summary_df.to_csv(TAB_DIR / 'tab1_summary.csv')
summary_df

### 1.6  Conclusion by asset class

- **Equities**: ARCH effects present in both periods. ACF(1) of squared returns has
  declined post-COVID for most indices — volatility shocks resolve more quickly. Fat tails
  persist; Jarque-Bera rejects normality in both periods.

- **Commodities**: Oil futures show the most dramatic change — ARCH effects strengthen
  and kurtosis increases sharply, reflecting the April 2020 disruption and 2022 energy
  shock. Gold is more stable.

- **Credit**: Strong ARCH effects in both periods. ACF(1) declines post-COVID, consistent
  with faster repricing of credit risk in a higher-rate environment.

- **FX**: Weaker ARCH effects than other classes. USDCHF is the exception: no ARCH
  effects pre-COVID (SNB peg) but significant ARCH effects post-COVID.

- **Overall**: volatility clustering is present in all asset classes and both periods. The
  stylized facts have changed in magnitude but not in nature — GARCH remains appropriate.
  The stationarity analysis (§0.5) confirms that log-returns are integrated of order 0,
  satisfying the key prerequisite for GARCH estimation.

---
## Section 2 — Conditional volatility (GARCH models)

Section 1 confirms time-varying, persistent volatility across all asset classes. We now
model conditional volatility using GARCH(1,1) estimated separately on pre-COVID and
post-COVID periods, and formally test whether the parameters — and in particular
persistence — have changed.

### 2.1  The GARCH(1,1) model

The GARCH(1,1) model of Bollerslev (1986) specifies:

$$\sigma^2_t = \omega + \alpha\varepsilon^2_{t-1} + \beta\sigma^2_{t-1}$$

where $\varepsilon_t = r_t - \mu$, $\omega > 0$, $\alpha \geq 0$, $\beta \geq 0$,
$\alpha + \beta < 1$.

The sum $\alpha + \beta$ is the **persistence** parameter — close to 1 means highly
persistent shocks.

Half-life of a volatility shock: $h = \log(0.5) / \log(\alpha + \beta)$

**Why GARCH(1,1) and not higher-order models?** The empirical literature (Hansen and
Lunde, 2005) shows that GARCH(1,1) is not significantly outperformed by GARCH(p,q)
with p,q > 1 for daily financial returns in terms of out-of-sample forecasting accuracy.
Its parsimony (3 parameters) is preferred for the pre/post-COVID subsamples which cover
approximately 2,500 and 1,600 observations respectively.

**Why not only GJR-GARCH?** We estimate GARCH(1,1) as the baseline for comparability
across all 12 assets, then introduce GJR-GARCH (Section 2.13) specifically for asset
classes where significant skewness (S&P 500: −1.36 pre-COVID) suggests leverage effects.

The model is estimated using `differential_evolution` as a global pre-optimizer followed
by multi-start L-BFGS-B refinement, ensuring robustness against local optima.

In [ ]:
def garch_recursion(params, eps):
    omega, alpha, beta = params
    T = len(eps)
    sigma2 = np.empty(T)
    sigma2[0] = np.var(eps)
    for t in range(1, T):
        sigma2[t] = omega + alpha * eps[t-1]**2 + beta * sigma2[t-1]
    return sigma2

def garch_loglik(params, eps):
    omega, alpha, beta = params
    if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 1:
        return 1e10
    sigma2 = garch_recursion(params, eps)
    if np.any(sigma2 <= 0):
        return 1e10
    return 0.5 * np.sum(np.log(sigma2) + eps**2 / sigma2)

def fit_garch(series):
    eps  = (series - series.mean()).values
    var0 = np.var(eps)
    bounds_de = [(1e-8, var0), (1e-6, 0.5), (0.5, 0.9999)]
    result_de = differential_evolution(
        garch_loglik, bounds_de, args=(eps,), seed=42,
        maxiter=1000, tol=1e-10, popsize=15, mutation=(0.5, 1), recombination=0.7,
    )
    starts = [result_de.x,
              [var0*0.05, 0.10, 0.85],
              [var0*0.02, 0.05, 0.92],
              [var0*0.01, 0.08, 0.90]]
    bounds_l = [(1e-8, None), (1e-6, 0.5), (1e-6, 0.9999)]
    best, best_val = None, 1e10
    for s in starts:
        if s[1] + s[2] >= 1:
            continue
        res = minimize(garch_loglik, s, args=(eps,), method='L-BFGS-B', bounds=bounds_l,
                       options={'maxiter': 5000, 'ftol': 1e-14, 'gtol': 1e-10})
        if res.fun < best_val:
            best_val = res.fun
            best = res
    omega, alpha, beta = best.x
    T, k = len(eps), 3
    loglik = -garch_loglik(best.x, eps)
    return {
        'omega': omega, 'alpha': alpha, 'beta': beta,
        'persistence': alpha + beta,
        'half_life'  : np.log(0.5) / np.log(alpha + beta),
        'sigma2'     : garch_recursion(best.x, eps),
        'eps': eps, 'loglik': loglik,
        'AIC': 2*k - 2*loglik, 'BIC': np.log(T)*k - 2*loglik,
        'params': best.x, 'success': best.success,
    }

print('GARCH functions defined.')

### 2.2  Conditional volatility — full sample

GARCH(1,1) applied to the full sample (from 2010) for all assets per class.
Analysis restricted to 2010–2026.

In [ ]:
for cat, assets in CATEGORIES.items():
    available = [a for a in assets if a in returns.columns]
    n_assets  = len(available)
    fig, axes = plt.subplots(n_assets, 1, figsize=(14, 4*n_assets), sharex=True)
    if n_assets == 1:
        axes = [axes]
    for ax, asset in zip(axes, available):
        series = returns.loc[PRE_START:][asset].dropna()
        g   = fit_garch(series)
        vol = pd.Series(np.sqrt(g['sigma2']) * np.sqrt(ANNUALIZATION) * 100, index=series.index)
        vol_pre  = vol[vol.index <  COVID_BREAK]
        vol_post = vol[vol.index >= COVID_BREAK]
        ax.fill_between(vol_pre.index,  vol_pre,  alpha=0.4, color=COLORS['pre'])
        ax.fill_between(vol_post.index, vol_post, alpha=0.4, color=COLORS['post'])
        ax.plot(vol_pre,  linewidth=0.8, color=COLORS['pre'],  label='Pre-COVID')
        ax.plot(vol_post, linewidth=0.8, color=COLORS['post'], label='Post-COVID')
        ax.axvline(pd.Timestamp(COVID_BREAK), color='black', linestyle='--', linewidth=1.2)
        ax.set_title(asset)
        ax.set_ylabel('Vol (%)')
        ax.legend(fontsize=8)
    fig.suptitle(f'GARCH(1,1) conditional volatility — {cat}', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'fig2_garch_vol_{cat.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

### 2.3  CUSUM test — statistical validation of the March 2020 breakpoint

The CUSUM test on squared demeaned returns provides a model-free test for variance
instability over the full 2010–2026 window. A crossing of the 5% critical bound (±1.36)
signals a structural change in the second moment.

**Note on the proxy**: the CUSUM is applied to squared demeaned returns rather than
GARCH-standardised residuals. This is a deliberate choice: at this stage of the analysis,
no model has yet been fitted on the full sample. The squared demeaned return
$\varepsilon^2_t = (r_t - \bar{r})^2$ is an unbiased estimator of $\sigma^2_t$ and is
model-free. Its use here is consistent with the objective of validating the breakpoint
without conditioning on a specific volatility specification. This limitation is
acknowledged in Section 5.4.

In [ ]:
def cusum_test(series):
    eps2 = (series - series.mean())**2
    mean_eps2 = eps2.mean()
    std_eps2  = eps2.std(ddof=1)
    T = len(eps2)
    cusum = np.cumsum((eps2.values - mean_eps2) / (std_eps2 * np.sqrt(T)))
    bound = 1.36
    crossed   = np.any(np.abs(cusum) > bound)
    cross_idx = np.where(np.abs(cusum) > bound)[0]
    cross_date = series.index[cross_idx[0]] if len(cross_idx) > 0 else None
    return cusum, bound, crossed, cross_date

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
cusum_results = []
for idx, (cat, asset) in enumerate(representatives.items()):
    full_series = returns.loc[PRE_START:][asset].dropna()
    cusum, bound, crossed, cross_date = cusum_test(full_series)
    ax = axes[idx]
    ax.plot(full_series.index, cusum, linewidth=0.8, color=COLORS['full'], label='CUSUM')
    ax.axhline( bound, color='red', linestyle='--', linewidth=1, label=f'±{bound} (5%)')
    ax.axhline(-bound, color='red', linestyle='--', linewidth=1)
    ax.axhline(0, color='black', linewidth=0.5, linestyle=':')
    ax.axvline(pd.Timestamp(COVID_BREAK), color='firebrick', linestyle='--',
               linewidth=1.5, label='COVID break')
    if cross_date is not None:
        ax.axvline(cross_date, color='orange', linestyle=':', linewidth=1.5,
                   label=f'First crossing: {cross_date.date()}')
    ax.set_title(f'{cat} — {asset}\nBreak detected: {crossed}')
    ax.set_ylabel('CUSUM statistic')
    ax.legend(fontsize=7)
    cusum_results.append({
        'Asset class': cat, 'Asset': asset,
        'Break detected (5%)': crossed,
        'First crossing date' : cross_date.date() if cross_date else None,
        'Max |CUSUM|'         : round(np.max(np.abs(cusum)), 3),
    })
fig.suptitle('CUSUM test on squared returns — variance stability (2010–2026)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_cusum.png', dpi=150, bbox_inches='tight')
plt.show()
cusum_df = pd.DataFrame(cusum_results)
cusum_df.to_csv(TAB_DIR / 'tab2_cusum.csv', index=False)
print(cusum_df.to_string(index=False))

### 2.4  GARCH parameters — pre vs post-COVID

We estimate GARCH(1,1) models separately on the pre-COVID (2010–2020) and post-COVID
(2020–2026) samples for all 12 assets.

**Two numerical edge cases flagged explicitly:**

1. **IGARCH boundary (α+β = 1.000)**: US HY Bonds (pre-COVID) and USDCHF (pre-COVID)
   reach the unit-root boundary, yielding infinite theoretical half-lives. This is
   consistent with near-integrated volatility — the unconditional variance is technically
   undefined, but the conditional process remains well-specified.

2. **Degrees of freedom at bound (ν = 6.00)**: several assets in the Student-t GARCH
   hit the upper bound of ν imposed in estimation (Section 2.8). Results should be
   treated as lower-bound estimates.

In [ ]:
garch_results = []
garch_fits    = {}
for asset in returns.columns:
    cat = next((c for c, lst in CATEGORIES.items() if asset in lst), 'Other')
    garch_fits[asset] = {}
    for label, ret in [('Pre-COVID', ret_pre), ('Post-COVID', ret_post)]:
        series = ret[asset].dropna()
        g = fit_garch(series)
        garch_fits[asset][label] = g
        uncond_var = (g['omega'] / (1 - g['persistence'])
                      if g['persistence'] < 1 else np.nan)
        garch_results.append({
            'Asset class'        : cat, 'Asset': asset, 'Period': label,
            'omega (×1e4)'       : round(g['omega'] * 1e4, 4),
            'alpha'              : round(g['alpha'], 4),
            'beta'               : round(g['beta'],  4),
            'alpha+beta'         : round(g['persistence'], 4),
            'half-life (days)'   : round(g['half_life'], 1),
            'Uncond. Ann. Vol (%)': (round(np.sqrt(uncond_var * ANNUALIZATION) * 100, 2)
                                     if not np.isnan(uncond_var) else np.nan),
            'AIC': round(g['AIC'], 1), 'BIC': round(g['BIC'], 1),
        })
garch_df = pd.DataFrame(garch_results).set_index(['Asset class', 'Asset', 'Period'])
garch_df.to_csv(TAB_DIR / 'tab2_garch_params.csv')
garch_df

### 2.5  Likelihood-Ratio test — formal test of parameter stability

H₀: GARCH parameters (ω, α, β) are identical pre- and post-COVID (3 constraints, χ²(3)).

$$LR = 2(\ell_{\text{unres}} - \ell_{\text{res}})$$

**Numerical note on the max(..., 0) correction**: for several assets the raw LR statistic
is slightly negative. Mathematically the unrestricted log-likelihood cannot be lower than
the restricted one; this violation arises from numerical optimisation — the two separate
optimisers and the pooled optimiser start from different initial conditions and converge
to different local optima.

To mitigate this, we **initialise the pooled model at the inverse-variance-weighted
average of the pre and post parameter estimates**, giving the pooled optimiser a better
starting point. For assets where a small negative value persists despite this, max(LR, 0)
is applied as a conservative correction (resulting in p-value = 1.0).

In [ ]:
lr_results = []
for asset in returns.columns:
    cat   = next((c for c, lst in CATEGORIES.items() if asset in lst), 'Other')
    g_pre  = garch_fits[asset]['Pre-COVID']
    g_post = garch_fits[asset]['Post-COVID']
    ll_unres = g_pre['loglik'] + g_post['loglik']

    # Pooled model — warm-started at weighted average of sub-period estimates
    w_pre  = len(ret_pre[asset].dropna())
    w_post = len(ret_post[asset].dropna())
    w_tot  = w_pre + w_post
    warm_start = (np.array(g_pre['params'])  * w_pre  +
                  np.array(g_post['params']) * w_post) / w_tot

    series_pooled = pd.concat([ret_pre[asset], ret_post[asset]]).dropna()
    eps_pooled = (series_pooled - series_pooled.mean()).values
    var0 = np.var(eps_pooled)
    bounds_de = [(1e-8, var0), (1e-6, 0.5), (0.5, 0.9999)]
    result_de = differential_evolution(
        garch_loglik, bounds_de, args=(eps_pooled,), seed=42,
        maxiter=1000, tol=1e-10, popsize=15,
    )
    starts_pool = [warm_start, result_de.x,
                   [var0*0.05, 0.10, 0.85], [var0*0.02, 0.05, 0.92]]
    bounds_l = [(1e-8, None), (1e-6, 0.5), (1e-6, 0.9999)]
    best_pool, best_val_pool = None, 1e10
    for s in starts_pool:
        if s[1] + s[2] >= 1:
            continue
        res = minimize(garch_loglik, s, args=(eps_pooled,), method='L-BFGS-B',
                       bounds=bounds_l,
                       options={'maxiter': 5000, 'ftol': 1e-14, 'gtol': 1e-10})
        if res.fun < best_val_pool:
            best_val_pool = res.fun
            best_pool = res
    ll_res = -best_val_pool

    lr_stat  = max(2 * (ll_unres - ll_res), 0)
    p_value  = 1 - chi2.cdf(lr_stat, df=3)
    lr_results.append({
        'Asset class'     : cat, 'Asset': asset,
        'LL unrestricted' : round(ll_unres, 2),
        'LL restricted'   : round(ll_res,   2),
        'LR statistic'    : round(lr_stat,  3),
        'p-value (χ²(3))' : round(p_value,  4),
        'Reject H0 (5%)'  : 'Yes ✓' if p_value < 0.05 else 'No ✗',
    })
lr_df = pd.DataFrame(lr_results).set_index(['Asset class', 'Asset'])
lr_df.to_csv(TAB_DIR / 'tab2_lr_test.csv')
print('H0: GARCH parameters identical pre- and post-COVID (3 constraints, χ²(3))')
print(f'Critical value at 5%: {chi2.ppf(0.95, df=3):.3f}\n')
lr_df

In [ ]:
ordered_assets_s2 = [a for cat in CATEGORIES for a in CATEGORIES[cat] if a in returns.columns]
fig, ax = plt.subplots(figsize=(14, 5))
x, width = np.arange(len(ordered_assets_s2)), 0.38
p_pre, p_post = [], []
for a in ordered_assets_s2:
    p_pre.append(garch_fits[a]['Pre-COVID']['persistence'])
    p_post.append(garch_fits[a]['Post-COVID']['persistence'])
ax.bar(x - width/2, p_pre,  width, label='Pre-COVID (2010-2020)',  color=COLORS['pre'])
ax.bar(x + width/2, p_post, width, label='Post-COVID (2020-2026)', color=COLORS['post'])
ax.axhline(1, color='black', linewidth=0.8, linestyle='--', label='Unit root')
for i, a in enumerate(ordered_assets_s2):
    cat_a = next((c for c, l in CATEGORIES.items() if a in l), 'Other')
    if lr_df.loc[(cat_a, a), 'Reject H0 (5%)'] == 'Yes ✓':
        ax.text(i, max(p_pre[i], p_post[i]) + 0.004, '✓', ha='center', fontsize=10)
boundaries, count = [], 0
for cat, assets in CATEGORIES.items():
    count += len([a for a in assets if a in returns.columns])
    boundaries.append(count - 0.5)
for b in boundaries[:-1]:
    ax.axvline(b, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
labels = [f'{a}\n({next((c for c, l in CATEGORIES.items() if a in l), "")})' for a in ordered_assets_s2]
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Persistence (α + β)')
ax.set_title('GARCH(1,1) persistence — pre vs post-COVID  [✓ = LR test significant at 5%]')
ax.set_ylim(0.82, 1.02)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_garch_persistence.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.7  Variance equality tests — Levene and Fisher F-test

The LR test above focuses on the shape of the volatility process (ω, α, β jointly). As a
complementary test targeting the unconditional variance directly, we apply:

- **Levene test** (Brown-Forsythe variant, `centre='median'`): robust to non-normality,
  tests H₀: equal variance pre vs post.
- **Fisher F-test**: classical ratio of sample variances.

A series is declared changed if **either** test rejects H₀ at 5%, following the
conservative approach of Levene (1960).

In [ ]:
var_test_results = []
for asset in returns.columns:
    cat   = next((c for c, lst in CATEGORIES.items() if asset in lst), 'Other')
    s_pre  = ret_pre[asset].dropna()
    s_post = ret_post[asset].dropna()
    lev_stat, lev_p = levene(s_pre, s_post, center='median')
    var_pre   = s_pre.var(ddof=1)
    var_post  = s_post.var(ddof=1)
    f_stat    = var_pre / var_post if var_post > 0 else np.nan
    df1, df2  = len(s_pre) - 1, len(s_post) - 1
    f_p       = 2 * min(f_dist.cdf(f_stat, df1, df2), 1 - f_dist.cdf(f_stat, df1, df2))
    var_test_results.append({
        'Asset class'      : cat, 'Asset': asset,
        'Var Pre (ann %)'  : round(np.sqrt(var_pre  * ANNUALIZATION) * 100, 2),
        'Var Post (ann %)' : round(np.sqrt(var_post * ANNUALIZATION) * 100, 2),
        'Levene stat'      : round(lev_stat, 3),
        'Levene p-val'     : round(lev_p,    4),
        'F-stat'           : round(f_stat,   3),
        'F p-val'          : round(f_p,      4),
        'Var changed (5%)' : 'Yes ✓' if lev_p < 0.05 else 'No ✗',
    })
var_df = pd.DataFrame(var_test_results).set_index(['Asset class', 'Asset'])
var_df.to_csv(TAB_DIR / 'tab2_variance_tests.csv')
print('H0: Equal unconditional variance pre- and post-COVID\n')
var_df

### 2.8  Student-t GARCH — conditional tail thickness

The Gaussian GARCH(1,1) may underestimate tail risk if standardised residuals remain
leptokurtic. We extend to Student-t innovations with estimated degrees of freedom ν.
Lower ν indicates fatter conditional tails beyond what GARCH volatility dynamics
already capture.

Note: the degrees-of-freedom parameter is bounded above at ν_max = 2.05 + e^5 ≈ 150
during estimation. Assets reporting exactly ν ≈ 6.00 have hit the effective upper bound
of the `log_nu` parameterisation and their reported ν should be treated as a lower-bound
estimate rather than a precise value.

In [ ]:
def garch_loglik_student(params, eps):
    omega, alpha, beta, log_nu = params
    nu = 2.05 + np.exp(log_nu)
    if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 1:
        return 1e10
    sigma2 = garch_recursion([omega, alpha, beta], eps)
    if np.any(sigma2 <= 0):
        return 1e10
    z = eps / np.sqrt(sigma2)
    log_dens = (gammaln((nu+1)/2) - gammaln(nu/2)
                - 0.5*np.log(np.pi*(nu-2))
                - ((nu+1)/2)*np.log(1 + z**2/(nu-2))
                - 0.5*np.log(sigma2))
    return -np.sum(log_dens)

def fit_garch_student(series):
    eps = (series - series.mean()).values
    g   = fit_garch(series)
    x0  = [g['omega'], g['alpha'], g['beta'], np.log(6 - 2.05)]
    bounds = [(1e-8, None), (1e-6, 0.5), (1e-6, 0.9999), (-2, 5)]
    res = minimize(garch_loglik_student, x0, args=(eps,), method='L-BFGS-B', bounds=bounds,
                   options={'maxiter': 5000, 'ftol': 1e-14, 'gtol': 1e-10})
    omega, alpha, beta, log_nu = res.x
    nu = 2.05 + np.exp(log_nu)
    T, k = len(eps), 4
    loglik = -garch_loglik_student(res.x, eps)
    return {'omega': omega, 'alpha': alpha, 'beta': beta, 'nu': nu,
            'persistence': alpha + beta,
            'sigma2': garch_recursion([omega, alpha, beta], eps),
            'eps': eps, 'loglik': loglik,
            'AIC': 2*k - 2*loglik, 'BIC': np.log(T)*k - 2*loglik}

student_results = []
for asset in returns.columns:
    cat = next((c for c, lst in CATEGORIES.items() if asset in lst), 'Other')
    for label, ret in [('Pre-COVID', ret_pre), ('Post-COVID', ret_post)]:
        series = ret[asset].dropna()
        g = fit_garch_student(series)
        student_results.append({
            'Asset class': cat, 'Asset': asset, 'Period': label,
            'alpha+beta' : round(g['persistence'], 4),
            'nu'         : round(g['nu'], 2),
        })
student_df = pd.DataFrame(student_results).set_index(['Asset class', 'Asset', 'Period'])
student_df.to_csv(TAB_DIR / 'tab2_student_params.csv')
student_df

### 2.9  Forecast evaluation — Diebold-Mariano test (out-of-sample)

We compare Gaussian vs Student-t GARCH using a true out-of-sample design: parameters
estimated on the first 80% of each sub-period, forecasts generated recursively on the
remaining 20%. The squared return serves as the variance proxy.

Two loss functions: MSE and QLIKE (Patton, 2011 — more robust to outliers in variance
forecasting).

- DM stat > 0 → Student-t has lower loss (preferred)
- |DM| > 1.96 → significant at 5%

Note on USDCHF pre-COVID: DM ≈ 14.8 is numerically anomalous. The SNB peg during this
period produces near-zero variance in the test set, causing the denominator of the DM
statistic to approach zero. These values are excluded from comparative interpretation.

In [ ]:
def forecast_garch_oos(params, eps_train, eps_test):
    omega, alpha, beta = params
    sigma2_train = garch_recursion(params, eps_train)
    sigma2_last  = sigma2_train[-1]
    T_test = len(eps_test)
    sigma2_oos = np.empty(T_test)
    sigma2_oos[0] = omega + alpha * eps_train[-1]**2 + beta * sigma2_last
    for t in range(1, T_test):
        sigma2_oos[t] = omega + alpha * eps_test[t-1]**2 + beta * sigma2_oos[t-1]
    return sigma2_oos

def forecast_garch_student_oos(params, eps_train, eps_test):
    omega, alpha, beta, _ = params
    return forecast_garch_oos([omega, alpha, beta], eps_train, eps_test)

def mse_loss(proxy, forecast):
    return (proxy - forecast)**2

def qlike_loss(proxy, forecast):
    return np.log(np.maximum(forecast, 1e-12)) + proxy / np.maximum(forecast, 1e-12)

def dm_stat(loss1, loss2):
    d   = loss1 - loss2
    std = d.std(ddof=1)
    if std == 0 or np.isnan(std):
        return np.nan
    return d.mean() / (std / np.sqrt(len(d)))

dm_results = []
for asset in returns.columns:
    cat = next((c for c, lst in CATEGORIES.items() if asset in lst), 'Other')
    for label, ret in [('Pre-COVID', ret_pre), ('Post-COVID', ret_post)]:
        series = ret[asset].dropna()
        if len(series) < 100:
            continue
        n = len(series)
        split = int(n * 0.8)
        train, test = series.iloc[:split], series.iloc[split:]
        eps_train = (train - train.mean()).values
        eps_test  = (test  - train.mean()).values
        g_g = fit_garch(train)
        g_s = fit_garch_student(train)
        s2_g = forecast_garch_oos(g_g['params'], eps_train, eps_test)
        s2_s = forecast_garch_student_oos(
            [g_s['omega'], g_s['alpha'], g_s['beta'],
             np.log(max(g_s['nu'] - 2.05, 1e-6))],
            eps_train, eps_test,
        )
        proxy  = eps_test**2
        dm_mse   = dm_stat(pd.Series(mse_loss(proxy, s2_g)),
                           pd.Series(mse_loss(proxy, s2_s)))
        dm_qlike = dm_stat(pd.Series(qlike_loss(proxy, s2_g)),
                           pd.Series(qlike_loss(proxy, s2_s)))
        dm_results.append({
            'Asset class': cat, 'Asset': asset, 'Period': label,
            'DM stat (MSE)'  : round(dm_mse,   3),
            'DM stat (QLIKE)': round(dm_qlike,  3),
            'Preferred (MSE)'  : 'Student-t' if dm_mse   > 0 else 'Gaussian',
            'Preferred (QLIKE)': 'Student-t' if dm_qlike > 0 else 'Gaussian',
        })
dm_df = pd.DataFrame(dm_results).set_index(['Asset class', 'Asset', 'Period'])
dm_df.to_csv(TAB_DIR / 'tab2_dm_test.csv')
print('DM stat > 0: Student-t has lower loss. |DM| > 1.96: significant at 5%.')
print('Note: USDCHF pre-COVID DM values are anomalous (near-zero variance under SNB peg).')
dm_df

### 2.10  Residual diagnostics — all assets

Ljung-Box test on squared standardised residuals $z^2_t = \varepsilon^2_t / \hat{\sigma}^2_t$.
Failure to reject H₀ (p > 0.05) indicates no remaining ARCH effects — the model has
adequately captured the volatility dynamics.

In [ ]:
diag_results = []
for asset in returns.columns:
    cat = next((c for c, lst in CATEGORIES.items() if asset in lst), 'Other')
    for label, ret in [('Pre-COVID', ret_pre), ('Post-COVID', ret_post)]:
        series = ret[asset].dropna()
        g = fit_garch(series)
        z = g['eps'] / np.sqrt(g['sigma2'])
        lb = acorr_ljungbox(z**2, lags=[10], return_df=True)
        diag_results.append({
            'Asset class'  : cat, 'Asset': asset, 'Period': label,
            'LB(10) stat z²' : round(lb['lb_stat'].values[0],   2),
            'LB(10) p-val z²': round(lb['lb_pvalue'].values[0], 4),
            'Model OK (5%)' : 'Yes ✓' if lb['lb_pvalue'].values[0] > 0.05 else 'No ✗',
        })
diag_df = pd.DataFrame(diag_results).set_index(['Asset class', 'Asset', 'Period'])
diag_df.to_csv(TAB_DIR / 'tab2_residual_diagnostics.csv')
diag_df

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
rep_assets = {'Equities': 'S&P500', 'Commodities': 'Oil futures',
              'Credit': 'US HY Bonds', 'FX': 'EURUSD'}
for row, (cat, asset) in enumerate(rep_assets.items()):
    for col, (label, ret) in enumerate([('Pre-COVID (2010-2020)', ret_pre),
                                        ('Post-COVID (2020-2026)', ret_post)]):
        series = ret[asset].dropna()
        g = fit_garch(series)
        z = g['eps'] / np.sqrt(g['sigma2'])
        color = COLORS['pre'] if col == 0 else COLORS['post']
        x_range = np.linspace(-6, 6, 300)
        axes[row, col*2].hist(z, bins=60, density=True, alpha=0.5, color=color)
        axes[row, col*2].plot(x_range, norm.pdf(x_range), 'k--', linewidth=1, label='N(0,1)')
        axes[row, col*2].set_xlim(-6, 6)
        axes[row, col*2].set_title(f'{asset}\n{label}', fontsize=8)
        axes[row, col*2].legend(fontsize=7)
        acf_vals = [pd.Series(z**2).autocorr(lag=i) for i in range(1, 21)]
        conf = 1.96 / np.sqrt(len(z))
        axes[row, col*2+1].bar(range(1, 21), acf_vals, color=color, alpha=0.7)
        axes[row, col*2+1].axhline( conf, linestyle='--', color='black', linewidth=0.8)
        axes[row, col*2+1].axhline(-conf, linestyle='--', color='black', linewidth=0.8)
        lb = acorr_ljungbox(z**2, lags=[10], return_df=True)
        axes[row, col*2+1].set_title(
            f'ACF(z²) — {label}\nLB(10) p={lb["lb_pvalue"].values[0]:.3f}', fontsize=8)
        axes[row, col*2+1].set_xlabel('Lag', fontsize=7)
fig.suptitle('GARCH(1,1) residual diagnostics — one representative per class', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_garch_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.12  GJR-GARCH — asymmetric volatility (leverage effects)

The descriptive statistics in Section 0 reveal significant negative skewness in equity
and credit returns (S&P500: −1.36 pre-COVID, Eurostoxx 50: −0.75, US HY Bonds: −4.32).
This asymmetry is consistent with the **leverage effect** documented by Black (1976) and
Christie (1982): negative return shocks increase financial leverage, raising future
volatility more than positive shocks of equal magnitude.

The symmetric GARCH(1,1) misspecifies this asymmetry — it treats $\varepsilon_t^2$ and
$(-\varepsilon_t)^2$ identically, biasing upwards the estimates of α and β for assets
with strong leverage effects.

We extend to the **GJR-GARCH(1,1)** model of Glosten, Jagannathan and Runkle (1993):

$$\sigma^2_t = \omega + (\alpha + \gamma \mathbb{1}_{\varepsilon_{t-1}<0})\varepsilon^2_{t-1} + \beta\sigma^2_{t-1}$$

where $\gamma \geq 0$ captures the additional volatility from negative shocks. The
asymmetry coefficient γ is tested individually (H₀: γ = 0). We restrict analysis to
**Equities and Credit** where the leverage effect is theoretically motivated and the
descriptive statistics most clearly indicate asymmetry.

In [ ]:
def gjr_garch_recursion(params, eps):
    omega, alpha, gamma, beta = params
    T = len(eps)
    sigma2 = np.empty(T)
    sigma2[0] = np.var(eps)
    for t in range(1, T):
        indicator = 1.0 if eps[t-1] < 0 else 0.0
        sigma2[t] = (omega
                     + (alpha + gamma * indicator) * eps[t-1]**2
                     + beta * sigma2[t-1])
    return sigma2

def gjr_loglik(params, eps):
    omega, alpha, gamma, beta = params
    # Stationarity: E[alpha + gamma/2 + beta] < 1
    if omega <= 0 or alpha < 0 or gamma < 0 or beta < 0:
        return 1e10
    if alpha + gamma/2 + beta >= 1:
        return 1e10
    sigma2 = gjr_garch_recursion(params, eps)
    if np.any(sigma2 <= 0):
        return 1e10
    return 0.5 * np.sum(np.log(sigma2) + eps**2 / sigma2)

def fit_gjr_garch(series):
    eps  = (series - series.mean()).values
    var0 = np.var(eps)
    # Initialise from symmetric GARCH estimate
    g0 = fit_garch(series)
    x0 = [g0['omega'], g0['alpha'] * 0.7, g0['alpha'] * 0.3, g0['beta']]
    bounds = [(1e-8, None), (1e-6, 0.5), (0.0, 0.5), (1e-6, 0.9999)]
    res = minimize(gjr_loglik, x0, args=(eps,), method='L-BFGS-B', bounds=bounds,
                   options={'maxiter': 5000, 'ftol': 1e-14, 'gtol': 1e-10})
    if not res.success:
        # Fallback: differential evolution
        bounds_de = [(1e-8, var0*0.1), (1e-6, 0.3), (0.0, 0.4), (0.5, 0.9999)]
        res_de = differential_evolution(gjr_loglik, bounds_de, args=(eps,),
                                        seed=42, maxiter=500, tol=1e-10)
        x0_2   = res_de.x
        res = minimize(gjr_loglik, x0_2, args=(eps,), method='L-BFGS-B', bounds=bounds,
                       options={'maxiter': 5000, 'ftol': 1e-14, 'gtol': 1e-10})
    omega, alpha, gamma_gjr, beta = res.x
    T, k = len(eps), 4
    loglik = -gjr_loglik(res.x, eps)
    persistence_gjr = alpha + gamma_gjr/2 + beta  # correct persistence for GJR
    # LR test: H0: gamma = 0 (symmetric, i.e. standard GARCH(1,1))
    g_sym  = fit_garch(series)
    lr_gjr = max(2 * (loglik - g_sym['loglik']), 0)
    p_gamma = 1 - chi2.cdf(lr_gjr, df=1)   # 1 additional parameter
    return {
        'omega': omega, 'alpha': alpha, 'gamma': gamma_gjr, 'beta': beta,
        'persistence': persistence_gjr,
        'half_life'  : np.log(0.5) / np.log(persistence_gjr) if persistence_gjr < 1 else np.nan,
        'sigma2'     : gjr_garch_recursion(res.x, eps),
        'eps': eps, 'loglik': loglik,
        'AIC': 2*k - 2*loglik, 'BIC': np.log(T)*k - 2*loglik,
        'LR vs GARCH': round(lr_gjr, 3),
        'p-value (gamma=0)': round(p_gamma, 4),
        'Asymmetry significant': 'Yes ✓' if p_gamma < 0.05 else 'No ✗',
    }

# Apply GJR-GARCH to Equities and Credit
gjr_assets = ([a for a in CATEGORIES['Equities'] if a in returns.columns]
              + [a for a in CATEGORIES['Credit']  if a in returns.columns])

gjr_results = []
gjr_fits    = {}
for asset in gjr_assets:
    cat = next((c for c, lst in CATEGORIES.items() if asset in lst), 'Other')
    gjr_fits[asset] = {}
    for label, ret in [('Pre-COVID', ret_pre), ('Post-COVID', ret_post)]:
        series = ret[asset].dropna()
        g = fit_gjr_garch(series)
        gjr_fits[asset][label] = g
        gjr_results.append({
            'Asset class'            : cat, 'Asset': asset, 'Period': label,
            'omega (×1e4)'           : round(g['omega'] * 1e4,  4),
            'alpha'                  : round(g['alpha'],         4),
            'gamma'                  : round(g['gamma'],         4),
            'beta'                   : round(g['beta'],          4),
            'Persistence (α+γ/2+β)'  : round(g['persistence'],  4),
            'half-life (days)'        : round(g['half_life'],    1) if not np.isnan(g['half_life']) else np.nan,
            'LR vs GARCH(1,1)'       : g['LR vs GARCH'],
            'p-value (γ=0)'          : g['p-value (gamma=0)'],
            'Asymmetry significant'  : g['Asymmetry significant'],
        })
gjr_df = pd.DataFrame(gjr_results).set_index(['Asset class', 'Asset', 'Period'])
gjr_df.to_csv(TAB_DIR / 'tab2_gjr_garch.csv')
gjr_df

**Interpretation of GJR-GARCH results.** A significant γ (p < 0.05) confirms that
negative shocks generate more volatility than positive shocks of equal magnitude — the
leverage effect. Where γ is significant, the true conditional persistence under the GJR
model (α + γ/2 + β) may differ from the GARCH(1,1) estimate, and the GARCH(1,1)
persistence reported in §2.4 should be read with this caveat.

The GJR specification reduces mis-specification of the volatility process for affected
assets. Its omission in the symmetric GARCH(1,1) does not invalidate the LR test in §2.5
(which is an internal test within the GARCH(1,1) family) but may affect the precise
numerical estimates of persistence. This limitation is acknowledged in §5.4.

### 2.13  Conclusion by asset class

- **Equities**: LR tests reject parameter stability for most indices (p < 0.05).
  Persistence has decreased — volatility shocks dissipate faster post-COVID. Student-t
  confirms fatter conditional tails. GJR-GARCH confirms significant leverage effects
  (γ > 0) pre-COVID for most equity indices. Levene tests confirm higher unconditional
  volatility. **Changed.**

- **Commodities**: Oil futures show near-integrated pre-COVID volatility (α+β ≈ 0.998).
  Despite lower post-COVID persistence, unconditional variance is higher because ω has
  risen substantially — the April 2020 disruption and 2022 energy shock permanently
  elevated the baseline. CUSUM detects a clear break. **Changed.**

- **Credit**: LR tests show mixed results. GJR-GARCH reveals significant leverage effects
  in US HY Bonds pre-COVID, consistent with the asymmetric credit spread widening during
  stress periods. US IG Bonds show slightly increased persistence post-COVID, consistent
  with the persistent repricing of interest rate risk during the 2021–2022 inflation
  shock. **Partially changed.**

- **FX**: LR tests fail to reject parameter stability for EURUSD and USDJPY. USDCHF is
  the exception (SNB peg removal dominant effect). **Largely stable, USDCHF excepted.**

---
## Section 3 — Dynamic correlations and portfolio risk

Section 2 showed that the level and persistence of volatility has changed since COVID-19,
heterogeneously across asset classes. But volatility alone does not tell the full story.
A portfolio manager cares about co-movement: how assets move together, whether
diversification still works, and how the covariance structure has evolved.

This section addresses the second dimension of risk structure: **risk co-movement**.

### 3.1  Rolling correlations — pre vs post-COVID

In [ ]:
ROLLING_WINDOW     = 252
returns_restricted = returns.loc[PRE_START:]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
for idx, (cat, assets) in enumerate(CATEGORIES.items()):
    available = [a for a in assets if a in returns_restricted.columns]
    if len(available) < 2:
        axes[idx].set_visible(False)
        continue
    avg_corr, index = [], []
    for i in range(ROLLING_WINDOW - 1, len(returns_restricted)):
        window = returns_restricted[available].iloc[i - ROLLING_WINDOW + 1 : i + 1]
        corr = window.corr().values
        n = corr.shape[0]
        avg_corr.append(corr[np.triu_indices(n, 1)].mean())
        index.append(returns_restricted.index[i])
    avg_corr = pd.Series(avg_corr, index=index)
    ac_pre   = avg_corr[avg_corr.index <  COVID_BREAK]
    ac_post  = avg_corr[avg_corr.index >= COVID_BREAK]
    ax = axes[idx]
    ax.plot(ac_pre,  linewidth=0.8, color=COLORS['pre'],  label='Pre-COVID')
    ax.plot(ac_post, linewidth=0.8, color=COLORS['post'], label='Post-COVID')
    ax.axvline(pd.Timestamp(COVID_BREAK), color='black', linestyle='--', linewidth=1.2)
    ax.axhline(ac_pre.mean(),  color=COLORS['pre'],  linestyle='--', linewidth=0.8,
               label=f'Pre mean: {ac_pre.mean():.2f}')
    ax.axhline(ac_post.mean(), color=COLORS['post'], linestyle='--', linewidth=0.8,
               label=f'Post mean: {ac_post.mean():.2f}')
    ax.set_title(f'Average pairwise correlation — {cat}')
    ax.set_ylabel('Correlation')
    ax.legend(fontsize=7)
fig.suptitle('Rolling within-class correlations (252-day window)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_rolling_corr_within.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2  Cross-asset class correlation matrix — pre vs post-COVID

In [ ]:
cats  = list(CATEGORIES.keys())
n_cat = len(cats)

def avg_cross_corr(ret, cat1, cat2):
    a1 = [a for a in CATEGORIES[cat1] if a in ret.columns]
    a2 = [a for a in CATEGORIES[cat2] if a in ret.columns]
    corrs = [ret[a].corr(ret[b]) for a in a1 for b in a2 if a != b]
    return np.mean(corrs) if corrs else np.nan

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, ret) in zip(axes, [('Pre-COVID (2010-2020)', ret_pre),
                                    ('Post-COVID (2020-2026)', ret_post)]):
    matrix = np.zeros((n_cat, n_cat))
    for i, c1 in enumerate(cats):
        for j, c2 in enumerate(cats):
            if i == j:
                avail = [a for a in CATEGORIES[c1] if a in ret.columns]
                if len(avail) >= 2:
                    corr = ret[avail].corr().values
                    n = corr.shape[0]
                    matrix[i, j] = corr[np.triu_indices(n, 1)].mean()
                else:
                    matrix[i, j] = 1.0
            else:
                matrix[i, j] = avg_cross_corr(ret, c1, c2)
    im = ax.imshow(matrix, cmap='RdYlGn', vmin=-0.5, vmax=0.5)
    ax.set_xticks(range(n_cat)); ax.set_yticks(range(n_cat))
    ax.set_xticklabels(cats, rotation=30, ha='right', fontsize=9)
    ax.set_yticklabels(cats, fontsize=9)
    ax.set_title(label)
    for i in range(n_cat):
        for j in range(n_cat):
            ax.text(j, i, f'{matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=axes[1], label='Average pairwise correlation')
fig.suptitle('Cross-asset class correlation matrix — pre vs post-COVID', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_corr_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

corr_change = pd.DataFrame(index=cats, columns=cats, dtype=float)
for i, c1 in enumerate(cats):
    for j, c2 in enumerate(cats):
        if i != j:
            pre_v  = avg_cross_corr(ret_pre,  c1, c2)
            post_v = avg_cross_corr(ret_post, c1, c2)
        else:
            avail_pre  = [a for a in CATEGORIES[c1] if a in ret_pre.columns]
            avail_post = [a for a in CATEGORIES[c1] if a in ret_post.columns]
            c_pre  = ret_pre[avail_pre].corr().values   if len(avail_pre)  >= 2 else np.array([[1]])
            c_post = ret_post[avail_post].corr().values if len(avail_post) >= 2 else np.array([[1]])
            n_p, n_po = c_pre.shape[0], c_post.shape[0]
            pre_v  = c_pre[np.triu_indices(n_p, 1)].mean()   if n_p  >= 2 else np.nan
            post_v = c_post[np.triu_indices(n_po, 1)].mean() if n_po >= 2 else np.nan
        corr_change.loc[c1, c2] = round(post_v - pre_v, 3)
print('Change in average correlation (Post - Pre):')
corr_change.to_csv(TAB_DIR / 'tab3_corr_change.csv')
corr_change

### 3.2b  Formal test of correlation matrix equality — Jennrich (1970) test

The visual comparison above suggests that correlation matrices have shifted. We formalise
this with the **Jennrich (1970) test**, which tests H₀: R_pre = R_post under the
assumption of multivariate normality. The test statistic follows a χ²(k(k-1)/2)
distribution where k is the number of assets.

For asset classes with k > 2, we also report the **Fisher z-test** on individual pairs,
with Bonferroni correction for multiple comparisons within each class.

In [ ]:
def jennrich_test(ret1, ret2):
    # Jennrich (1970) chi-square test for equality of two correlation matrices.
    # H0: R1 = R2.  Returns: statistic, p-value, degrees of freedom.
    n1, n2 = len(ret1), len(ret2)
    R1 = ret1.corr().values
    R2 = ret2.corr().values
    k  = R1.shape[0]
    # Pooled correlation
    S  = (n1 * R1 + n2 * R2) / (n1 + n2)
    S_inv = np.linalg.inv(S)
    # Jennrich statistic: chi2 approximation
    diff = R1 - R2
    Z = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            Z[i, j] = diff[i, j] / np.sqrt((S_inv[i,i]*S_inv[j,j] + S_inv[i,j]**2)
                                             * (1/n1 + 1/n2))
    # Test statistic: sum of squared z-scores over upper triangle (i < j)
    idx = np.triu_indices(k, 1)
    chi2_stat = np.sum(Z[idx]**2)
    dof       = k * (k - 1) // 2
    p_val     = 1 - chi2.cdf(chi2_stat, df=dof)
    return round(chi2_stat, 3), round(p_val, 4), dof

def fisher_z_test(r1, r2, n1, n2):
    # Two-sample Fisher z-test for equality of two Pearson correlations.
    z1 = np.arctanh(np.clip(r1, -0.9999, 0.9999))
    z2 = np.arctanh(np.clip(r2, -0.9999, 0.9999))
    se = np.sqrt(1/(n1-3) + 1/(n2-3))
    z_stat = (z1 - z2) / se
    p_val  = 2 * (1 - norm.cdf(abs(z_stat)))
    return round(z_stat, 3), round(p_val, 4)

jennrich_results = []
for cat, assets in CATEGORIES.items():
    available = [a for a in assets if a in returns.columns]
    if len(available) < 2:
        continue
    chi2_stat, p_val, dof = jennrich_test(ret_pre[available], ret_post[available])
    jennrich_results.append({
        'Asset class': cat,
        'k (assets)' : len(available),
        'df'         : dof,
        'χ² stat'    : chi2_stat,
        'p-value'    : p_val,
        'Reject H0 (5%)': 'Yes ✓' if p_val < 0.05 else 'No ✗',
    })

# Pairwise Fisher z-tests with Bonferroni correction
fisher_results = []
n_pre, n_post = len(ret_pre), len(ret_post)
for cat, assets in CATEGORIES.items():
    available = [a for a in assets if a in returns.columns]
    pairs_in_class = [(available[i], available[j])
                      for i in range(len(available))
                      for j in range(i+1, len(available))]
    n_pairs = len(pairs_in_class)
    alpha_bonf = 0.05 / max(n_pairs, 1)
    for (a, b) in pairs_in_class:
        r_pre  = ret_pre[a].corr(ret_pre[b])
        r_post = ret_post[a].corr(ret_post[b])
        z_stat, p_val = fisher_z_test(r_pre, r_post, n_pre, n_post)
        fisher_results.append({
            'Asset class': cat,
            'Pair'       : f'{a} / {b}',
            'r pre'      : round(r_pre,  3),
            'r post'     : round(r_post, 3),
            'Δr'         : round(r_post - r_pre, 3),
            'Fisher z'   : z_stat,
            'p-value'    : p_val,
            'α (Bonf.)'  : round(alpha_bonf, 4),
            'Significant (Bonf.)': 'Yes ✓' if p_val < alpha_bonf else 'No ✗',
        })

jenn_df   = pd.DataFrame(jennrich_results).set_index('Asset class')
fisher_df = pd.DataFrame(fisher_results).set_index(['Asset class', 'Pair'])
jenn_df.to_csv(TAB_DIR  / 'tab3_jennrich_test.csv')
fisher_df.to_csv(TAB_DIR / 'tab3_fisher_z_test.csv')
print('=== Jennrich test: H0: correlation matrix is identical pre- and post-COVID ===')
display(jenn_df)
print('\n=== Pairwise Fisher z-test (Bonferroni-corrected within asset class) ===')
fisher_df

### 3.3  Interest rate analysis

The 2022 rate hiking cycle is a key driver of post-COVID correlation changes,
particularly for Credit. We document the shift in yield levels and volatility, and
examine the equity-bond correlation dynamic directly.

The mechanism follows Shiller and Beltratti (1992) and Campbell and Ammer (1993):
in a discounted cash-flow framework, rising yields increase the discount rate for equity
and bond cash flows simultaneously, inducing a positive correlation between equity and
bond returns — reversing the traditional flight-to-quality negative correlation observed
in the 2010s low-rate environment.

In [ ]:
yield_changes = yields.diff().dropna() * 100  # basis points
ych_pre  = yield_changes.loc[PRE_START : pd.Timestamp(COVID_BREAK) - pd.Timedelta(days=1)]
ych_post = yield_changes.loc[COVID_BREAK:]

yield_stats = pd.DataFrame({
    'Pre-COVID Ann. Std (bp×√252)' : (ych_pre.std()  * np.sqrt(ANNUALIZATION)).round(3),
    'Post-COVID Ann. Std (bp×√252)': (ych_post.std() * np.sqrt(ANNUALIZATION)).round(3),
    'Pre-COVID Mean change (bp/day)': ych_pre.mean().round(3),
    'Post-COVID Mean change (bp/day)': ych_post.mean().round(3),
    'Yield level Pre end (%)'  : yields.loc[:pd.Timestamp(COVID_BREAK)].iloc[-1].round(3),
    'Yield level Post avg (%)' : yields.loc[COVID_BREAK:].mean().round(3),
}).T
yield_stats.to_csv(TAB_DIR / 'tab3c_yield_stats.csv')
print(yield_stats.to_string())

sp500_ret = returns['S&P500'].dropna()
ig_ret    = returns['US IG Bonds'].dropna()
common_idx = sp500_ret.index.intersection(ig_ret.index)
sp500_ret  = sp500_ret.loc[common_idx]
ig_ret     = ig_ret.loc[common_idx]
roll_corr_eb     = sp500_ret.rolling(126).corr(ig_ret)
us_yield_aligned = yields['US T 10-year Yield'].reindex(common_idx).ffill()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.plot(roll_corr_eb.loc['2010':], color=COLORS['full'], linewidth=0.8,
         label='S&P500 / US IG Bonds rolling 126d corr')
ax1.axhline(0, color='black', linewidth=0.5, linestyle=':')
ax1.axvline(pd.Timestamp(COVID_BREAK), color='firebrick', linestyle='--', linewidth=1.5)
ax1.set_ylabel('Rolling correlation (S&P500 / US IG Bonds)', color=COLORS['full'])
ax1.set_ylim(-1, 1)
ax2.plot(us_yield_aligned.loc['2010':], color='darkorange', linewidth=0.8, alpha=0.7,
         label='US 10Y yield')
ax2.set_ylabel('US 10-year yield (%)', color='darkorange')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='lower right')
plt.title('Equity-bond correlation vs US 10Y yield level\n'
          '(Rate hiking cycle of 2022 reversed the equity-bond diversification benefit)',
          fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3c_equity_bond_yield.png', dpi=150, bbox_inches='tight')
plt.show()

common_eb     = roll_corr_eb.dropna()
aligned_yield = us_yield_aligned.reindex(common_eb.index).dropna()
common2 = common_eb.index.intersection(aligned_yield.index)
pearson_r, pearson_p = stats.pearsonr(aligned_yield.loc[common2], common_eb.loc[common2])
print(f'Pearson correlation (yield level vs equity-bond corr): r={pearson_r:.3f}, p={pearson_p:.4f}')
print('Positive r: higher rates → more positive equity-bond correlation')
print('(diversification benefit disappears in high-rate environments)')

### 3.4  Equal Risk Contribution portfolio

The ERC portfolio allocates weights such that each asset contributes equally to total
portfolio risk (Maillard, Roncalli and Teïletche, 2010):

$$w_i \cdot \frac{\partial \sigma_p}{\partial w_i} = \frac{\sigma_p}{n} \quad \forall i$$

Unlike Markowitz, ERC requires no return forecasts and is driven entirely by the
covariance matrix. Changes in ERC weights across periods therefore directly measure
whether the risk structure has changed in a way that is **economically relevant** for
portfolio construction — not just statistically significant.

In [ ]:
def portfolio_volatility(w, cov):
    return float(np.sqrt(w @ cov @ w))

def risk_contributions(w, cov):
    sigma    = portfolio_volatility(w, cov)
    marginal = cov @ w / sigma
    total    = w * marginal
    return total, total / sigma

def erc_weights(cov, start=None):
    n   = cov.shape[0]
    cov = np.asarray(cov, dtype=float)
    cov = (cov + cov.T) / 2
    eig = np.linalg.eigvalsh(cov)
    if eig.min() <= 1e-12:
        cov = cov + np.eye(n) * (1e-8 - eig.min())
    if start is None:
        start = np.repeat(1/n, n)
    def objective(w):
        _, pct = risk_contributions(w, cov)
        return np.sum((pct - 1/n)**2)
    result = minimize(objective, start, method='SLSQP',
                      bounds=[(0, 1)]*n,
                      constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
                      options={'maxiter': 1000, 'ftol': 1e-12})
    if not result.success:
        result = minimize(objective, np.repeat(1/n, n), method='SLSQP',
                          bounds=[(0, 1)]*n,
                          constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
                          options={'maxiter': 2000, 'ftol': 1e-12})
    w = np.maximum(result.x, 0)
    return w / w.sum()

assets_list      = list(returns.columns)
ROLLING_COV_WINDOW = 252 * 2
month_ends = returns.resample('ME').last().index

erc_weights_ts, previous = [], None
for date in month_ends:
    if date < returns.index[ROLLING_COV_WINDOW]:
        continue
    loc = returns.index.searchsorted(date, side='right') - 1
    if loc < ROLLING_COV_WINDOW:
        continue
    cov = returns.iloc[loc - ROLLING_COV_WINDOW + 1 : loc + 1].cov().values * ANNUALIZATION
    w = erc_weights(cov, previous)
    previous = w
    erc_weights_ts.append(pd.Series(w, index=assets_list, name=returns.index[loc]))

erc_weights_df = pd.DataFrame(erc_weights_ts)
print(f'ERC weights computed: {len(erc_weights_df)} monthly observations')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.stackplot(erc_weights_df.index,
             [erc_weights_df[a] for a in assets_list],
             labels=assets_list, alpha=0.85)
ax.axvline(pd.Timestamp(COVID_BREAK), color='black', linestyle='--', linewidth=1.5)
ax.set_title('ERC portfolio weights — rolling 2-year covariance')
ax.set_ylabel('Portfolio weight')
ax.set_ylim(0, 1)
ax.legend(loc='upper left', ncol=3, fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_erc_weights.png', dpi=150, bbox_inches='tight')
plt.show()

erc_pre  = erc_weights_df[erc_weights_df.index <  COVID_BREAK].mean()
erc_post = erc_weights_df[erc_weights_df.index >= COVID_BREAK].mean()
erc_comparison = pd.DataFrame({
    'Asset class'         : [next((c for c, l in CATEGORIES.items() if a in l), 'Other')
                              for a in erc_weights_df.columns],
    'Pre-COVID avg weight' : erc_pre.round(3),
    'Post-COVID avg weight': erc_post.round(3),
    'Change'               : (erc_post - erc_pre).round(3),
})
erc_comparison.to_csv(TAB_DIR / 'tab3_erc_weights.csv')
erc_comparison

### 3.5b  ERC performance backtest — decision relevance

The statistical evidence for a change in risk structure is necessary but not sufficient.
Here we test **decision relevance**: does the covariance shift translate into meaningfully
different portfolio outcomes? We compare the rolling ERC portfolio against an equal-weight
benchmark over the full sample.

In [ ]:
daily_weights  = erc_weights_df.reindex(returns.index, method='ffill').dropna()
aligned_returns = returns.loc[daily_weights.index]
erc_ret   = (daily_weights * aligned_returns).sum(axis=1)
equal_ret = aligned_returns.mean(axis=1)

cumulative = pd.DataFrame({'Equal Weight': (1 + equal_ret).cumprod(),
                            'ERC'         : (1 + erc_ret).cumprod()})
fig, ax = plt.subplots(figsize=(13, 4))
(cumulative / cumulative.iloc[0]).plot(ax=ax)
ax.axvline(pd.Timestamp(COVID_BREAK), color='firebrick', linestyle='--', linewidth=1.5)
ax.set_title('ERC vs Equal-Weight: cumulative performance')
ax.set_ylabel('Growth of 1')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_erc_vs_ew.png', dpi=150, bbox_inches='tight')
plt.show()

def performance_statistics(r):
    ann_return   = (1 + r).prod() ** (ANNUALIZATION / len(r)) - 1
    ann_vol      = r.std() * np.sqrt(ANNUALIZATION)
    sharpe       = ann_return / ann_vol
    wealth       = (1 + r).cumprod()
    max_drawdown = (wealth / wealth.cummax() - 1).min()
    return pd.Series({'Ann. return' : round(ann_return,   4),
                      'Ann. vol'    : round(ann_vol,       4),
                      'Sharpe'      : round(sharpe,        3),
                      'Max drawdown': round(max_drawdown,  4)})

perf_stats = pd.DataFrame({'Equal Weight': performance_statistics(equal_ret),
                            'ERC'         : performance_statistics(erc_ret)}).T
for period_label, mask in [('Pre-COVID',  erc_ret.index <  COVID_BREAK),
                            ('Post-COVID', erc_ret.index >= COVID_BREAK)]:
    print(f'\n--- {period_label} ---')
    print(pd.DataFrame({'Equal Weight': performance_statistics(equal_ret[mask]),
                        'ERC'         : performance_statistics(erc_ret[mask])}).T.to_string())
perf_stats.to_csv(TAB_DIR / 'tab3_erc_performance.csv')
print('\n--- Full sample ---')
perf_stats

### 3.6  DCC filter — dynamic correlations

The Dynamic Conditional Correlation (DCC) model of Engle and Sheppard (2001) allows the
correlation matrix to vary over time:

$$Q_t = (1-\alpha-\beta)\bar{Q} + \alpha u_{t-1}u^{\top}_{t-1} + \beta Q_{t-1}$$
$$R_t = \text{diag}(Q_t)^{-1/2} Q_t \text{diag}(Q_t)^{-1/2}$$

where $u_t$ are EWMA-standardised residuals and $\bar{Q}$ is the unconditional
covariance of $u_t$.

**Why DCC and not CCC or O-GARCH?** The Constant Conditional Correlation model (Bollerslev,
1990) constrains $R_t = R$ for all t — inconsistent with the evidence from Section 3.1
showing time-varying rolling correlations. The Orthogonal GARCH of Alexander (2001)
requires principal component extraction which complicates economic interpretation.
DCC strikes the best balance between flexibility and parsimony.

**Implementation note**: the DCC parameters α and β are fixed at (0.02, 0.97) rather
than estimated via MLE. This is a deliberate simplification that trades estimation
efficiency for computational tractability across 12 assets simultaneously. The robustness
check in Section 3.9 with an alternative parameterisation (α=0.05, β=0.94) confirms that
all directional conclusions are parameter-insensitive.

In [ ]:
def dcc_filter(ret, lam_vol=0.94, alpha=0.02, beta=0.97, burn_in=252):
    X = ret.values.astype(float)
    T, n = X.shape
    mu  = X[:burn_in].mean(axis=0)
    eps = X - mu
    h   = np.full((T, n), np.nan)
    h[burn_in-1] = eps[:burn_in].var(axis=0, ddof=1)
    for t in range(burn_in, T):
        h[t] = lam_vol * h[t-1] + (1 - lam_vol) * eps[t-1]**2
    z = eps / np.sqrt(h)
    z[:burn_in] = np.nan
    Qbar = np.cov(z[burn_in:].T)
    Q    = Qbar.copy()
    H_list, R_list, dates = [], [], []
    for t in range(burn_in+1, T):
        u = z[t-1][:, None]
        Q = (1-alpha-beta)*Qbar + alpha*(u @ u.T) + beta*Q
        dQ = np.sqrt(np.diag(Q))
        R  = Q / np.outer(dQ, dQ)
        R  = np.clip(R, -0.999, 0.999)
        np.fill_diagonal(R, 1.0)
        D = np.diag(np.sqrt(h[t]))
        H_list.append(D @ R @ D * ANNUALIZATION)
        R_list.append(R.copy())
        dates.append(ret.index[t])
    return (pd.DatetimeIndex(dates), np.array(H_list), np.array(R_list),
            pd.DataFrame(h, index=ret.index, columns=ret.columns))

dcc_dates, Hs, Rs, h_dcc = dcc_filter(returns.loc[PRE_START:])
print(f'DCC filter (baseline: α=0.02, β=0.97): {len(dcc_dates)} observations')

In [ ]:
idx_map = {a: i for i, a in enumerate(assets_list)}
dcc_corr_results = []
for cat, assets in CATEGORIES.items():
    available = [a for a in assets if a in assets_list]
    if len(available) < 2:
        continue
    pairs = [(available[i], available[j])
             for i in range(len(available)) for j in range(i+1, len(available))]
    for a, b in pairs:
        corr_ts = pd.Series(Rs[:, idx_map[a], idx_map[b]], index=dcc_dates)
        pre_m   = corr_ts[corr_ts.index <  COVID_BREAK].mean()
        post_m  = corr_ts[corr_ts.index >= COVID_BREAK].mean()
        dcc_corr_results.append({
            'Asset class'   : cat, 'Pair': f'{a} / {b}',
            'DCC corr Pre'  : round(pre_m,          3),
            'DCC corr Post' : round(post_m,          3),
            'Change'        : round(post_m - pre_m,  3),
        })
dcc_corr_df = pd.DataFrame(dcc_corr_results).set_index(['Asset class', 'Pair'])
dcc_corr_df.to_csv(TAB_DIR / 'tab3_dcc_correlations.csv')
dcc_corr_df

In [ ]:
selected_pairs = {
    'Equities'   : ('S&P500',     'Eurostoxx 50'),
    'Commodities': ('Oil futures', 'Gold'),
    'Credit'     : ('US IG Bonds', 'US HY Bonds'),
    'FX'         : ('EURUSD',      'USDJPY'),
}
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
for ax_idx, (cat, (a, b)) in enumerate(selected_pairs.items()):
    corr_ts = pd.Series(Rs[:, idx_map[a], idx_map[b]], index=dcc_dates)
    cp      = corr_ts[corr_ts.index <  COVID_BREAK]
    cpo     = corr_ts[corr_ts.index >= COVID_BREAK]
    ax = axes[ax_idx]
    ax.plot(cp,  linewidth=0.8, color=COLORS['pre'],  label='Pre-COVID')
    ax.plot(cpo, linewidth=0.8, color=COLORS['post'], label='Post-COVID')
    ax.axvline(pd.Timestamp(COVID_BREAK), color='black', linestyle='--', linewidth=1.2)
    ax.axhline(cp.mean(),  color=COLORS['pre'],  linestyle='--', linewidth=0.8,
               label=f'Pre mean: {cp.mean():.2f}')
    ax.axhline(cpo.mean(), color=COLORS['post'], linestyle='--', linewidth=0.8,
               label=f'Post mean: {cpo.mean():.2f}')
    ax.set_title(f'{cat} — {a} / {b}')
    ax.set_ylabel('DCC correlation')
    ax.legend(fontsize=7)
fig.suptitle('DCC conditional correlations — selected pairs by asset class', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_dcc_corr_pairs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
dcc_dates_rob, _, Rs_rob, _ = dcc_filter(returns.loc[PRE_START:], lam_vol=0.94,
                                          alpha=0.05, beta=0.94)
rob_results = []
for cat, (a, b) in selected_pairs.items():
    corr_base = pd.Series(Rs[:, idx_map[a], idx_map[b]],     index=dcc_dates)
    corr_alt  = pd.Series(Rs_rob[:, idx_map[a], idx_map[b]], index=dcc_dates_rob)
    pre_b  = corr_base[corr_base.index <  COVID_BREAK].mean()
    post_b = corr_base[corr_base.index >= COVID_BREAK].mean()
    pre_a  = corr_alt[corr_alt.index   <  COVID_BREAK].mean()
    post_a = corr_alt[corr_alt.index   >= COVID_BREAK].mean()
    rob_results.append({
        'Asset class': cat, 'Pair': f'{a} / {b}',
        'Pre (α=0.02,β=0.97)'  : round(pre_b,  3),
        'Post (α=0.02,β=0.97)' : round(post_b, 3),
        'Pre (α=0.05,β=0.94)'  : round(pre_a,  3),
        'Post (α=0.05,β=0.94)' : round(post_a, 3),
        'Direction consistent' : '✓' if (post_b - pre_b) * (post_a - pre_a) > 0 else '✗',
    })
rob_df = pd.DataFrame(rob_results).set_index(['Asset class', 'Pair'])
rob_df.to_csv(TAB_DIR / 'tab3_dcc_robustness.csv')
print('DCC robustness: direction of change consistent across both parameterisations:')
rob_df

In [ ]:
dcc_erc_weights, prev_w = [], None
for date in month_ends:
    closest = dcc_dates[dcc_dates <= date]
    if len(closest) == 0:
        continue
    date_idx = np.searchsorted(dcc_dates, closest[-1])
    H = Hs[date_idx]
    eig = np.linalg.eigvalsh(H)
    if eig.min() <= 0:
        H = H + np.eye(len(assets_list)) * (1e-8 - eig.min())
    w = erc_weights(H, prev_w)
    prev_w = w
    dcc_erc_weights.append(pd.Series(w, index=assets_list, name=date))

dcc_erc_df   = pd.DataFrame(dcc_erc_weights)
dcc_erc_pre  = dcc_erc_df[dcc_erc_df.index <  COVID_BREAK].mean()
dcc_erc_post = dcc_erc_df[dcc_erc_df.index >= COVID_BREAK].mean()

comparison_df = pd.DataFrame({
    'Rolling ERC Pre' : erc_pre.round(3),
    'Rolling ERC Post': erc_post.round(3),
    'DCC ERC Pre'     : dcc_erc_pre.round(3),
    'DCC ERC Post'    : dcc_erc_post.round(3),
    'Rolling Δ'       : (erc_post     - erc_pre).round(3),
    'DCC Δ'           : (dcc_erc_post - dcc_erc_pre).round(3),
})
comparison_df.insert(0, 'Asset class',
    [next((c for c, l in CATEGORIES.items() if a in l), 'Other') for a in assets_list])
comparison_df.to_csv(TAB_DIR / 'tab3_erc_comparison.csv')
comparison_df

---
## Section 4 — Structural breaks and regime changes

Sections 2 and 3 documented changes in volatility levels and correlations. This section
addresses the third dimension: are these changes **permanent** (structural break) or
**recurring** (regime switch)? We use two complementary approaches: the Bai-Perron PELT
algorithm for endogenous break detection and a Markov-switching model for regime
identification.

### 4.1  Rolling correlations — the observable object

In [ ]:
ROLLING_WEEKS = 52
weekly_returns = returns.loc[PRE_START:].resample('W-FRI').last().dropna()

pairs = {
    'Equities'   : ('S&P500',     'Eurostoxx 50'),
    'Commodities': ('S&P500',     'Gold'),
    'Credit'     : ('S&P500',     'US IG Bonds'),
    'FX'         : ('S&P500',     'EURUSD'),
}
rolling_corrs = {}
for cat, (a, b) in pairs.items():
    rc = weekly_returns[a].rolling(ROLLING_WEEKS).corr(weekly_returns[b]).dropna()
    rolling_corrs[cat] = rc

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
for idx, (cat, rc) in enumerate(rolling_corrs.items()):
    a, b     = pairs[cat]
    rc_pre   = rc[rc.index <  COVID_BREAK]
    rc_post  = rc[rc.index >= COVID_BREAK]
    ax       = axes[idx]
    ax.plot(rc_pre,  linewidth=0.8, color=COLORS['pre'],  label='Pre-COVID')
    ax.plot(rc_post, linewidth=0.8, color=COLORS['post'], label='Post-COVID')
    ax.axvline(pd.Timestamp(COVID_BREAK), color='black', linestyle='--', linewidth=1.2)
    ax.axhline(rc_pre.mean(),  color=COLORS['pre'],  linestyle='--', linewidth=0.8,
               label=f'Pre mean: {rc_pre.mean():.2f}')
    ax.axhline(rc_post.mean(), color=COLORS['post'], linestyle='--', linewidth=0.8,
               label=f'Post mean: {rc_post.mean():.2f}')
    ax.axhline(0, color='black', linewidth=0.5, linestyle=':')
    ax.set_title(f'{cat} — {a} / {b}')
    ax.set_ylabel('52-week rolling correlation')
    ax.legend(fontsize=7)
fig.suptitle('52-week rolling correlations — key pairs by asset class', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_rolling_corr.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2  Markov-switching model on rolling correlations

A two-state Markov-switching model (Hamilton, 1989) is estimated on each rolling
correlation series. The two regimes — identified endogenously as low-correlation and
high-correlation states — capture the cyclical nature of co-movement.

A shift in the probability of being in the high-correlation regime post-COVID indicates
whether the change is regime-dependent or structural.

**Note on applying Markov-switching to rolling correlations**: rolling correlations are
smoothed estimators with overlapping windows, which introduces serial correlation in the
series itself (distinct from the underlying return process). This may bias the transition
probability estimates by inflating regime persistence. The PELT test in Section 4.3 is
applied directly to the rolling correlation series and provides a complementary
non-parametric perspective.

In [ ]:
ms_results = []
ms_fitted  = {}
for cat, rc in rolling_corrs.items():
    a, b = pairs[cat]
    ms_model  = sm.tsa.MarkovRegression(rc, k_regimes=2, trend='c',
                                         switching_variance=True)
    ms_result = ms_model.fit(disp=False, maxiter=300)
    probs = ms_result.smoothed_marginal_probabilities
    state_means = [(rc * probs[s]).sum() / probs[s].sum() for s in [0, 1]]
    high_state  = int(np.argmax(state_means))
    low_state   = 1 - high_state
    high_prob   = probs[high_state]
    prob_pre    = high_prob[high_prob.index <  COVID_BREAK].mean()
    prob_post   = high_prob[high_prob.index >= COVID_BREAK].mean()

    p_hh = np.nan
    for fmt in [f'p[{high_state}->{high_state}]',
                f'p[{high_state}][{high_state}]',
                f'p{high_state}{high_state}']:
        if fmt in ms_result.params.index:
            p_hh = ms_result.params[fmt]
            break
    if np.isnan(p_hh):
        try:
            trans = ms_result.transition_matrices
            p_hh  = float(trans[high_state, high_state, 0]) if trans.ndim == 3 else float(trans[high_state, high_state])
        except Exception:
            pass
    dur_high = 1 / (1 - p_hh) if not np.isnan(p_hh) and p_hh < 1 else np.nan

    ms_results.append({
        'Asset class'             : cat, 'Pair': f'{a} / {b}',
        'μ low-corr state'        : round(state_means[low_state],  3),
        'μ high-corr state'       : round(state_means[high_state], 3),
        'Avg duration high (weeks)': round(dur_high, 1) if not np.isnan(dur_high) else np.nan,
        'P(high) pre-COVID'       : round(prob_pre,  3),
        'P(high) post-COVID'      : round(prob_post, 3),
        'Change'                  : round(prob_post - prob_pre, 3),
    })
    ms_fitted[cat] = {'rc': rc, 'ms_result': ms_result,
                      'high_state': high_state, 'high_prob': high_prob}

ms_df = pd.DataFrame(ms_results).set_index(['Asset class', 'Pair'])
ms_df.to_csv(TAB_DIR / 'tab4_markov_switching.csv')
ms_df

### 4.2b  Significance of the two-state specification — LR test

Before interpreting regime probabilities, we test whether a two-state model is
statistically justified relative to a one-state model.

**Caveat**: under H₀ (k=1 regime), one transition probability parameter is on the
boundary of the parameter space, invalidating the standard χ² distribution for the LR
statistic. We follow Davies (1977) and use a **conservative upper bound**: p-value ≤ 2×
the standard χ²(1) p-value. This is the approach recommended by Hamilton (1994, p. 699)
for the specific case of testing k=1 vs k=2 regimes.

In [ ]:
lr_regimes_results = []
for cat, rc in rolling_corrs.items():
    a, b = pairs[cat]
    # 2-state model log-likelihood
    ll_2 = ms_fitted[cat]['ms_result'].llf

    # 1-state model: simple OLS (constant mean, constant variance)
    from statsmodels.regression.linear_model import OLS
    import statsmodels.formula.api as smf
    endog = rc.values
    X     = np.ones((len(endog), 1))
    ols   = OLS(endog, X).fit()
    ll_1  = ols.llf  # log-likelihood of single-state model

    lr_stat  = 2 * (ll_2 - ll_1)
    # Conservative Davies upper bound: p <= 2 * chi2(1) p-value
    p_chi2_1 = 1 - chi2.cdf(max(lr_stat, 0), df=1)
    p_davies  = min(2 * p_chi2_1, 1.0)
    lr_regimes_results.append({
        'Asset class'          : cat, 'Pair': f'{a} / {b}',
        'LL (1-state)'         : round(ll_1,     2),
        'LL (2-state)'         : round(ll_2,     2),
        'LR stat'              : round(lr_stat,   3),
        'p-value (Davies UB)'  : round(p_davies,  4),
        '2 states justified'   : 'Yes ✓' if p_davies < 0.05 else 'No ✗',
    })

lr_reg_df = pd.DataFrame(lr_regimes_results).set_index(['Asset class', 'Pair'])
lr_reg_df.to_csv(TAB_DIR / 'tab4_lr_regimes.csv')
print('LR test: H0 = 1 regime, H1 = 2 regimes. p-value: conservative Davies upper bound.')
lr_reg_df

### 4.3  Bai-Perron structural break detection — PELT algorithm

The PELT (Pruned Exact Linear Time) algorithm of Killick, Fearnhead and Eckley (2012)
detects multiple structural breaks in the mean of the rolling correlation series with
O(n) computational complexity. The penalty parameter is selected by BIC over a grid.
COVID is considered detected if any break date falls within ±1 year of 23 March 2020.

**Interpretation of break density**: PELT detects 11–12 breaks per series over a 15-year
window (~1/year). This is consistent with the known properties of rolling-window
correlation estimators — overlapping windows create smooth but autocorrelated series
where locally significant mean shifts (e.g. from a transient event) appear as formal
breaks. Each break captures a locally significant shift in the correlation series rather
than a structural change in the underlying return process. The COVID break is detected in
all four pairs, confirming robustness.

In [ ]:
bp_results = []
for cat, rc in rolling_corrs.items():
    a, b  = pairs[cat]
    y     = rc.values
    dates = rc.index
    model = rpt.Pelt(model='rbf', min_size=52, jump=1)
    model.fit(y)
    best_bic, best_breaks = np.inf, []
    for pen in [0.5, 1, 2, 3, 5, 10, 15, 20]:
        try:
            bkps        = model.predict(pen=pen)
            bkps_no_end = [b for b in bkps if b < len(y)]
            n_breaks    = len(bkps_no_end)
            residuals   = 0
            prev = 0
            for bk in bkps:
                seg = y[prev:bk]
                residuals += np.sum((seg - seg.mean())**2)
                prev = bk
            bic = len(y) * np.log(residuals / len(y)) + (n_breaks + 1) * np.log(len(y))
            if bic < best_bic:
                best_bic    = bic
                best_breaks = bkps_no_end
        except Exception:
            continue
    break_dates = [dates[i] for i in best_breaks if i < len(dates)]
    covid_detected = any(
        abs((pd.Timestamp(d) - pd.Timestamp(COVID_BREAK)).days) <= 365
        for d in break_dates) if break_dates else False
    bp_results.append({
        'Asset class'         : cat, 'Pair': f'{a} / {b}',
        'N breaks'            : len(break_dates),
        'Break dates'         : ', '.join([str(d.date()) for d in break_dates]),
        'COVID detected (±1yr)': '✓' if covid_detected else '✗',
    })
    ms_fitted[cat]['break_dates'] = break_dates

bp_df = pd.DataFrame(bp_results).set_index(['Asset class', 'Pair'])
bp_df.to_csv(TAB_DIR / 'tab4_bai_perron.csv')
bp_df

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=False)
for ax, (cat, data) in zip(axes, ms_fitted.items()):
    rc          = data['rc']
    high_prob   = data['high_prob']
    break_dates = data.get('break_dates', [])
    a, b        = pairs[cat]
    ax2 = ax.twinx()
    ax.fill_between(high_prob.index, high_prob, alpha=0.3, color='orange',
                    label='P(high-corr regime)')
    ax2.plot(rc, linewidth=0.8, color=COLORS['full'], alpha=0.8, label='52w rolling corr')
    ax2.axhline(0, color='black', linewidth=0.5, linestyle=':')
    ax.axvline(pd.Timestamp(COVID_BREAK), color='firebrick', linestyle='--',
               linewidth=1.5, label='COVID break (23 Mar 2020)')
    colors_bp = ['purple', 'green', 'navy']
    for k, bd in enumerate(break_dates):
        ax.axvline(bd, color=colors_bp[k % len(colors_bp)], linestyle=':', linewidth=1,
                   label=f'PELT break {k+1}: {bd.date()}')
    ax.set_ylim(0, 1)
    ax.set_ylabel('P(high-corr regime)')
    ax2.set_ylabel('Rolling correlation')
    ax.set_title(f'{cat} — {a} / {b}  ({len(break_dates)} PELT break(s))')
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc='upper left', ncol=2)
fig.suptitle('Markov-switching regimes and PELT break dates — by asset class', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_regimes_breaks.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.5  Conclusion by asset class

- **Equities**: Higher probability of high-correlation regime post-COVID. PELT detects a
  break close to the COVID date. LR test confirms two regimes are statistically justified.
  **Partially permanent change.**

- **Commodities**: S&P500/Gold correlation has shifted — Gold increasingly safe-haven,
  lower high-correlation regime probability post-COVID. **Favourable structural change.**

- **Credit**: S&P500/US IG Bonds correlation has turned positive post-COVID, confirmed by
  PELT. The primary driver is the 2022 rate hiking cycle (which reversed the traditional
  negative equity-bond correlation). **Changed permanently.**

- **FX**: Most stable. Balanced regime probabilities pre and post-COVID. **Largely stable.**

- **Overall**: The evidence favours a partial structural break. COVID has made
  high-volatility / high-correlation states more frequent and more persistent, but not
  permanent across all asset classes. The LR regime significance tests (§4.2b) confirm
  that the two-state specification is statistically supported in all four pairs.

---
## Section 5 — Discussion and conclusion

### 5.1  Synthesis by asset class

The three dimensions of risk — volatility levels, co-movement, and regimes — have been
tested separately in Sections 2, 3, and 4. We now bring the evidence together by asset
class.

**Equities**: All three dimensions of risk have changed since COVID-19. Volatility
persistence has decreased for most equity indices. GJR-GARCH confirms significant
leverage effects pre-COVID (consistent with the strong negative skewness). Within-class
DCC correlations have decreased slightly but cross-class co-movement with Credit has
increased. Bai-Perron detects structural breaks close to the COVID date, and the
Markov-switching model assigns higher probability to the high-correlation regime
post-COVID. The ERC portfolio allocates meaningfully less weight to equities post-COVID.
The Jennrich test formally rejects equality of the correlation matrix. **The structure of
risk in equities has changed.**

**Commodities**: Oil futures show the most dramatic change across all three dimensions.
Volatility has increased structurally (ω is permanently higher), persistence has
decreased, and CUSUM detects a clear break. Gold shows a more nuanced picture — its
safe-haven role has strengthened (lower correlation with equities post-COVID). **The
structure of risk in commodities has changed.**

**Credit**: The most economically significant change is in the equity-bond correlation.
The traditional negative S&P500/US IG Bonds correlation has turned positive post-COVID,
driven primarily by the 2022 rate hiking cycle. This is confirmed by the DCC filter,
PELT, and the Pearson correlation between yield levels and rolling equity-bond correlation
(r = 0.663, p < 0.001), consistent with the theoretical mechanism of Shiller and Beltratti
(1992). The Jennrich test rejects matrix equality. GJR-GARCH reveals significant leverage
effects in US HY Bonds. **Credit risk structure has partially changed, with the rate cycle
as primary driver.**

**FX**: The most stable asset class. LR tests fail to reject parameter stability for
EURUSD and USDJPY. DCC correlations show small changes directionally consistent with the
robustness check. USDCHF is the exception — the 2015 SNB peg removal dominates the
pre-COVID regime. **FX risk structure has largely not changed.**

### 5.2  Economic relevance — ERC portfolio implications

The ERC backtest confirms that the changes documented above are not only statistically
significant but also economically relevant. The post-COVID covariance matrix leads to a
meaningfully different allocation: FX weight (EURUSD, USDCHF) has increased substantially
(+0.298 combined) while Credit weight has fallen (−0.196 combined), reflecting the
simultaneous increase in Credit volatility and the loss of the equity-bond diversification
benefit. This reallocation confirms that pre-COVID covariance matrices are no longer valid
inputs for post-COVID portfolio construction.

### 5.3  Final answer to the research question

**Has the structure of risk in financial markets changed since COVID-19?**

The answer is **partially yes**, with significant heterogeneity across asset classes.

| Asset class | Volatility (LR + Levene) | Co-movement (DCC + Jennrich) | Regimes (PELT + MS + LR) | Overall |
|---|---|---|---|---|
| **Equities**    | Changed ✓   | Changed ✓     | Changed ✓   | **Yes** |
| **Commodities** | Changed ✓   | Changed ✓     | Changed ✓   | **Yes** |
| **Credit**      | Partial ~   | Changed ✓     | Changed ✓   | **Partially** |
| **FX**          | Stable ✗    | Stable ✗      | Stable ✗    | **No** |

The change is best characterised as a **partial structural break** rather than a pure
regime switch. The post-COVID regime has different parameters from the pre-COVID regime,
but the Markov-switching evidence (LR-tested in §4.2b) suggests that high-volatility,
high-correlation states are not entirely new — COVID has made them more frequent and more
persistent, but not permanent across all asset classes.

### 5.4  Limitations

1. **Sample size**: the post-COVID period covers approximately 1,600 observations
   (March 2020 to April 2026) vs ~2,700 pre-COVID. While sufficient for GARCH estimation,
   parameter estimates carry greater uncertainty for the post-COVID period.

2. **Fixed DCC parameters**: α and β are fixed at (0.02, 0.97) rather than estimated by
   MLE. The robustness check in Section 3.9 confirms directional consistency across two
   parameterisations, but formally estimated parameters (via the two-step Engle-Sheppard
   estimator) could yield more precise correlation dynamics.

3. **Breakpoint choice**: the March 2020 date is well-justified for Equities and
   Commodities. For Credit, the primary driver appears to be the rate hiking cycle
   beginning in late 2021, suggesting an alternative breakpoint would be more appropriate
   for that asset class.

4. **CUSUM proxy**: the CUSUM in Section 2.3 is applied to squared demeaned returns rather
   than GARCH-standardised residuals, making it a model-free but noisier variance
   stability test.

5. **Asymmetric GARCH scope**: GJR-GARCH is estimated only for Equities and Credit.
   Commodities (notably Gold) and some FX pairs may also exhibit asymmetric responses;
   extending the GJR analysis to all 12 assets is a natural extension.

6. **Markov-switching on rolling correlations**: as noted in §4.2, the smoothing
   introduced by rolling windows may inflate estimated regime persistence. Applying the
   Markov-switching model directly to daily returns would provide a cleaner specification.

7. **No external instruments**: VIX, inflation expectations, and policy rate paths are not
   included as explanatory variables. Their inclusion could improve the economic
   interpretation of the regime changes identified in Section 4.

8. **LR test numerical precision**: for several assets, the max(LR, 0) correction is
   applied due to optimisation noise in the pooled model. The warm-start initialisation in
   §2.5 mitigates this but does not eliminate it entirely. The directional evidence from
   the Levene test and the visual comparison of GARCH parameters remains valid
   independently.